#**Renovaí - análises**

##Extração da base

In [0]:

# Credenciais do banco (Secret Scope "renovai")
jdbcHostname = dbutils.secrets.get(scope="renovai", key="pg_host")
jdbcDatabase = dbutils.secrets.get(scope="renovai", key="pg_database")
jdbcPort = 5432

url = f"jdbc:postgresql://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}"

properties = {
    "user": dbutils.secrets.get(scope="renovai", key="pg_user"),
    "password": dbutils.secrets.get(scope="renovai", key="pg_password"),
    "driver": "org.postgresql.Driver"
}

# Query para pegar os nomes das tabelas
query = """
(
select table_name
from information_schema.tables
where table_schema = 'public'
and table_type = 'BASE TABLE'
and table_name not like 'auditoria%'
) as tabelas
"""

# Conectando com o banco
tabelas = spark.read.jdbc(
    url=url,
    table=query,
    properties=properties
)

# Dicionário de DataFrames
dfs = {}

for linha in tabelas.collect():
    nome_tabela = linha["table_name"]

    print(f"Lendo {nome_tabela}...")

    # eventos_operacionais é tabela-pai (INHERITS): sem ONLY o SELECT
    # devolve também as linhas de coletas e triagens
    fonte = (
        "(SELECT * FROM ONLY eventos_operacionais) AS t"
        if nome_tabela == "eventos_operacionais"
        else nome_tabela
    )

    dfs[nome_tabela] = spark.read.jdbc(
        url=url,
        table=fonte,
        properties=properties
    )

Lendo usuarios...
Lendo empresas...
Lendo empresa_materiais_interesses...
Lendo empresa_cooperativas_favoritas...
Lendo cargos...
Lendo status...
Lendo materiais...
Lendo perfis...
Lendo enderecos...
Lendo telefones...
Lendo equipes...
Lendo equipes_cooperados...
Lendo funcionarios...
Lendo pedidos_cooperativas...
Lendo pedido_itens...
Lendo negociacoes...
Lendo negociacao_mensagens...
Lendo avaliacoes...
Lendo eventos_operacionais...
Lendo coletas...
Lendo tipos_rateios...
Lendo estoques...
Lendo triagens...
Lendo movimentacoes_estoques...
Lendo negociacao_itens...
Lendo lancamentos_despesas...
Lendo rotas_enderecos...
Lendo rotas...
Lendo rateios_funcionarios...
Lendo despesas...
Lendo rateios...
Lendo categorias_materiais...
Lendo cooperativas...
Lendo pedidos...
Lendo rota_execucoes...
Lendo rota_execucoes_paradas...


##Tratamento de dados

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

# Chaves primárias conforme tables.sql (usadas para medir duplicidade real)
CHAVES_PRIMARIAS = {
    "enderecos": "endereco_id", "categorias_materiais": "categoria_id",
    "materiais": "material_id", "cooperativas": "cooperativa_id",
    "empresas": "empresa_id", "empresa_materiais_interesses": "empresa_material_id",
    "empresa_cooperativas_favoritas": "favorito_id", "perfis": "perfil_id",
    "usuarios": "usuario_id", "cargos": "cargo_id", "status": "status_id",
    "funcionarios": "funcionario_id", "telefones": "telefone_id",
    "equipes": "equipe_id", "equipes_cooperados": "equipe_cooperado_id",
    "pedidos": "pedido_id", "pedido_itens": "item_id",
    "pedidos_cooperativas": "pedido_cooperativa_id", "negociacoes": "negociacao_id",
    "negociacao_itens": "negociacao_item_id", "negociacao_mensagens": "mensagem_id",
    "avaliacoes": "avaliacao_id", "eventos_operacionais": "evento_id",
    "coletas": "evento_id", "triagens": "evento_id", "estoques": "estoque_id",
    "movimentacoes_estoques": "movimentacao_id", "tipos_rateios": "tipo_rateio_id",
    "rateios": "rateio_id", "rateios_funcionarios": "rateio_funcionario_id",
    "despesas": "despesa_id", "lancamentos_despesas": "lancamento_id",
    "rotas": "rota_id", "rotas_enderecos": "rota_endereco_id",
}

# simpleString() de NUMERIC é "decimal(p,s)": por isso o startswith
TIPOS_NUMERICOS = {"int", "bigint", "smallint", "tinyint", "float", "double"}

def eh_numerico(tipo_coluna):
    return tipo_coluna in TIPOS_NUMERICOS or tipo_coluna.startswith("decimal")

schema_catalogo = StructType([
    StructField("tabela", StringType(), True),
    StructField("coluna", StringType(), True),
    StructField("tipo", StringType(), True),
    StructField("quantidade_linhas", LongType(), True),
    StructField("quantidade_colunas", LongType(), True),
    StructField("quantidade_nulos", LongType(), True),
    StructField("percentual_nulos", DoubleType(), True),
    StructField("valores_distintos", LongType(), True),
    StructField("quantidade_duplicados", LongType(), True),
    StructField("percentual_duplicados", DoubleType(), True),
    StructField("minimo", DoubleType(), True),
    StructField("maximo", DoubleType(), True),
    StructField("media", DoubleType(), True),
    StructField("desvio_padrao", DoubleType(), True)
])


def gerar_catalogo(dfs_dict):
    """Um único job de agregação por tabela (antes: 2 jobs por coluna)."""
    catalogo = []

    for nome_tabela, df in dfs_dict.items():

        if nome_tabela.startswith("auditoria"):
            continue

        print(f"Analisando {nome_tabela}...")

        campos = df.schema.fields
        pk = CHAVES_PRIMARIAS.get(nome_tabela)
        tem_pk = pk is not None and pk in df.columns

        exprs = [F.count(F.lit(1)).alias("__linhas")]
        if tem_pk:
            exprs.append(F.countDistinct(pk).alias("__pk_distintas"))

        for i, campo in enumerate(campos):
            c = F.col(campo.name)
            exprs.append(F.sum(c.isNull().cast("int")).alias(f"n{i}"))
            exprs.append(F.countDistinct(c).alias(f"d{i}"))
            if eh_numerico(campo.dataType.simpleString()):
                cd = c.cast("double")
                exprs += [F.min(cd).alias(f"mi{i}"), F.max(cd).alias(f"ma{i}"),
                          F.avg(cd).alias(f"me{i}"), F.stddev(cd).alias(f"dp{i}")]

        r = df.agg(*exprs).collect()[0].asDict()

        qtd_linhas = int(r["__linhas"])
        # Duplicidade real = mesma chave primária repetida
        qtd_dup = qtd_linhas - int(r["__pk_distintas"]) if tem_pk else 0
        perc_dup = float(qtd_dup / qtd_linhas * 100) if qtd_linhas > 0 else 0.0

        for i, campo in enumerate(campos):
            qtd_nulos = int(r[f"n{i}"] or 0)
            num = eh_numerico(campo.dataType.simpleString())
            catalogo.append({
                "tabela": nome_tabela,
                "coluna": campo.name,
                "tipo": campo.dataType.simpleString(),
                "quantidade_linhas": qtd_linhas,
                "quantidade_colunas": len(campos),
                "quantidade_nulos": qtd_nulos,
                "percentual_nulos": float(qtd_nulos / qtd_linhas * 100) if qtd_linhas > 0 else 0.0,
                "valores_distintos": int(r[f"d{i}"]),
                "quantidade_duplicados": int(qtd_dup),
                "percentual_duplicados": perc_dup,
                "minimo": r.get(f"mi{i}") if num else None,
                "maximo": r.get(f"ma{i}") if num else None,
                "media": r.get(f"me{i}") if num else None,
                "desvio_padrao": r.get(f"dp{i}") if num else None,
            })

    return spark.createDataFrame(catalogo, schema=schema_catalogo)


catalogo_df = gerar_catalogo(dfs)

display(catalogo_df)

Analisando usuarios...
Analisando empresas...
Analisando empresa_materiais_interesses...
Analisando empresa_cooperativas_favoritas...
Analisando cargos...
Analisando status...
Analisando materiais...
Analisando perfis...
Analisando enderecos...
Analisando telefones...
Analisando equipes...
Analisando equipes_cooperados...
Analisando funcionarios...
Analisando pedidos_cooperativas...
Analisando pedido_itens...
Analisando negociacoes...
Analisando negociacao_mensagens...
Analisando avaliacoes...
Analisando eventos_operacionais...
Analisando coletas...
Analisando tipos_rateios...
Analisando estoques...
Analisando triagens...
Analisando movimentacoes_estoques...
Analisando negociacao_itens...
Analisando lancamentos_despesas...
Analisando rotas_enderecos...
Analisando rotas...
Analisando rateios_funcionarios...
Analisando despesas...
Analisando rateios...
Analisando categorias_materiais...
Analisando cooperativas...
Analisando pedidos...
Analisando rota_execucoes...
Analisando rota_execucoe

tabela,coluna,tipo,quantidade_linhas,quantidade_colunas,quantidade_nulos,percentual_nulos,valores_distintos,quantidade_duplicados,percentual_duplicados,minimo,maximo,media,desvio_padrao
usuarios,usuario_id,string,416,14,0,0.0,416,0,0.0,null,null,null,null
usuarios,nome,string,416,14,0,0.0,415,0,0.0,null,null,null,null
usuarios,email,string,416,14,0,0.0,416,0,0.0,null,null,null,null
usuarios,cpf,string,416,14,0,0.0,416,0,0.0,null,null,null,null
usuarios,senha,string,416,14,416,100.0,0,0,0.0,null,null,null,null
usuarios,senha_hash,string,416,14,0,0.0,416,0,0.0,null,null,null,null
usuarios,token_redefinicao,string,416,14,416,100.0,0,0,0.0,null,null,null,null
usuarios,imagem_url,string,416,14,413,99.27884615384616,2,0,0.0,null,null,null,null
usuarios,data_nascimento,date,416,14,9,2.1634615384615383,402,0,0.0,null,null,null,null
usuarios,ultimo_acesso,timestamp,416,14,398,95.67307692307693,18,0,0.0,null,null,null,null


In [0]:
catalogo_df = (
    catalogo_df
    .withColumn(
        "problemas",
        F.array_remove(
            F.array(
                F.when(
                    F.col("quantidade_nulos") > 0,
                    F.lit("NULOS")
                ),

                F.when(
                    F.col("quantidade_duplicados") > 0,
                    F.lit("DUPLICIDADE")
                ),

                F.when(
                    (F.col("desvio_padrao").isNotNull()) &
                    (F.col("media").isNotNull()) &
                    (F.col("desvio_padrao") > 0) &
                    (
                        F.abs(F.col("maximo") - F.col("media"))
                        > 3 * F.col("desvio_padrao")
                    ),
                    F.lit("POSSIVEL_OUTLIER")
                )
            ),
            None
        )
    )
)

display(catalogo_df)

tabela,coluna,tipo,quantidade_linhas,quantidade_colunas,quantidade_nulos,percentual_nulos,valores_distintos,quantidade_duplicados,percentual_duplicados,minimo,maximo,media,desvio_padrao,problemas
usuarios,usuario_id,string,416,14,0,0.0,416,0,0.0,null,null,null,null,null
usuarios,nome,string,416,14,0,0.0,415,0,0.0,null,null,null,null,null
usuarios,email,string,416,14,0,0.0,416,0,0.0,null,null,null,null,null
usuarios,cpf,string,416,14,0,0.0,416,0,0.0,null,null,null,null,null
usuarios,senha,string,416,14,416,100.0,0,0,0.0,null,null,null,null,null
usuarios,senha_hash,string,416,14,0,0.0,416,0,0.0,null,null,null,null,null
usuarios,token_redefinicao,string,416,14,416,100.0,0,0,0.0,null,null,null,null,null
usuarios,imagem_url,string,416,14,413,99.27884615384616,2,0,0.0,null,null,null,null,null
usuarios,data_nascimento,date,416,14,9,2.1634615384615383,402,0,0.0,null,null,null,null,null
usuarios,ultimo_acesso,timestamp,416,14,398,95.67307692307693,18,0,0.0,null,null,null,null,null


In [0]:
catalogo_df = catalogo_df.filter(~F.col("tabela").startswith("auditoria"))

### Tratamento geral

In [0]:
def padronizar_colunas(df):

    for coluna in df.columns:

        nova_coluna = (
            coluna
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        df = df.withColumnRenamed(
            coluna,
            nova_coluna
        )

    return df

In [0]:
query_tipos = """
(
    SELECT
        table_name,
        column_name,
        data_type,
        udt_name,
        numeric_precision,
        numeric_scale,
        character_maximum_length
    FROM information_schema.columns
    WHERE table_schema = 'public'
    ORDER BY table_name, ordinal_position
) AS tipos
"""

tipos_postgres = spark.read.jdbc(
    url=url,
    table=query_tipos,
    properties=properties
)

display(tipos_postgres)

table_name,column_name,data_type,udt_name,numeric_precision,numeric_scale,character_maximum_length
auditoria_avaliacoes,auditoria_id,uuid,uuid,null,null,null
auditoria_avaliacoes,registro_id,uuid,uuid,null,null,null
auditoria_avaliacoes,operacao,character varying,varchar,null,null,50
auditoria_avaliacoes,dados_antigos,jsonb,jsonb,null,null,null
auditoria_avaliacoes,dados_novos,jsonb,jsonb,null,null,null
auditoria_avaliacoes,campos_atualizados,ARRAY,_text,null,null,null
auditoria_avaliacoes,usuario_execucao,character varying,varchar,null,null,255
auditoria_avaliacoes,data_alteracao,timestamp without time zone,timestamp,null,null,null
auditoria_categorias_materiais,auditoria_id,uuid,uuid,null,null,null
auditoria_categorias_materiais,registro_id,uuid,uuid,null,null,null


In [0]:
from pyspark.sql.types import (
    StringType,
    BooleanType,
    IntegerType,
    LongType,
    ShortType,
    ByteType,
    FloatType,
    DoubleType,
    DecimalType,
    DateType,
    TimestampType
)
def postgres_para_spark(
    data_type,
    numeric_precision=None,
    numeric_scale=None
):

    if data_type in ["character varying", "varchar", "character", "char", "text"]:
        return StringType()

    elif data_type == "integer":
        return IntegerType()

    elif data_type == "bigint":
        return LongType()

    elif data_type == "smallint":
        return ShortType()

    elif data_type == "real":
        return FloatType()

    elif data_type == "double precision":
        return DoubleType()

    elif data_type == "numeric":

        if numeric_precision is not None and numeric_scale is not None:

            return DecimalType(
                int(numeric_precision),
                int(numeric_scale)
            )

        return DoubleType()

    elif data_type == "boolean":
        return BooleanType()

    elif data_type == "date":
        return DateType()

    elif data_type in [
        "timestamp without time zone",
        "timestamp with time zone"
    ]:
        return TimestampType()

    # UUID normalmente chega ao Spark como string
    elif data_type == "uuid":
        return StringType()

    # JSON/JSONB serão tratados como texto
    elif data_type in ["json", "jsonb"]:
        return StringType()

    else:
        return None

In [0]:
def tratar_tipos(df, nome_tabela, tipos_postgres):

    print(f"\nVerificando tipos: {nome_tabela}")

    # Tipos esperados somente da tabela atual
    tipos_tabela = (
        tipos_postgres
        .filter(
            F.col("table_name") == nome_tabela
        )
        .collect()
    )

    # Dicionário com os tipos do PostgreSQL
    tipos_esperados = {
        linha["column_name"]: linha
        for linha in tipos_tabela
    }

    alteracoes = []

    for campo in df.schema.fields:

        coluna = campo.name
        tipo_atual = campo.dataType

        # Se a coluna não estiver no PostgreSQL
        if coluna not in tipos_esperados:

            print(
                f"⚠️ {coluna}: "
                f"não encontrada no PostgreSQL"
            )

            continue

        info = tipos_esperados[coluna]

        # Tipo esperado
        tipo_esperado = postgres_para_spark(
            info["data_type"],
            info["numeric_precision"],
            info["numeric_scale"]
        )

        # Se não temos uma conversão definida
        if tipo_esperado is None:

            print(
                f"⚠️ {coluna}: "
                f"tipo PostgreSQL "
                f"{info['data_type']} "
                f"sem regra de conversão"
            )

            continue

        # ==========================================
        # COMPARAÇÃO
        # ==========================================

        tipo_atual_str = tipo_atual.simpleString()
        tipo_esperado_str = tipo_esperado.simpleString()

        print(
            f"{coluna}: "
            f"{tipo_atual_str} → "
            f"{tipo_esperado_str}"
        )

        # ==========================================
        # TIPOS DIFERENTES
        # ==========================================

        if tipo_atual != tipo_esperado:

            print(
                f"   ⚠️ Tipo diferente. "
                f"Convertendo..."
            )

            df = df.withColumn(
                coluna,
                F.col(coluna).cast(tipo_esperado)
            )

            alteracoes.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "tipo_atual": tipo_atual_str,
                "tipo_esperado": tipo_esperado_str,
                "acao": "CONVERTIDO"
            })

        else:

            alteracoes.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "tipo_atual": tipo_atual_str,
                "tipo_esperado": tipo_esperado_str,
                "acao": "OK"
            })

    return df, alteracoes

In [0]:
def tratar_nulos(df, catalogo_tabela):

    colunas_nulas = (
        catalogo_tabela
        .filter(F.col("quantidade_nulos") > 0)
        .select("coluna", "tipo")
        .collect()
    )

    for linha in colunas_nulas:

        coluna = linha["coluna"]
        tipo = linha["tipo"]

        # ======================================
        # CHAVES (UUID) E DATAS: NULL é informação
        # (ex.: data_conclusao NULL = pedido em aberto;
        #  triagem_id/item_id NULL definem a origem da movimentação;
        #  perfis/telefones dependem do XOR entre IDs)
        # ======================================

        if coluna.endswith("_id") or coluna.startswith("data_"):
            continue

        # ======================================
        # TEXTO DESCRITIVO
        # ======================================

        if tipo == "string":

            print(f"Tratando NULL: {coluna}")

            df = df.fillna({
                coluna: "Não informado"
            })

        # ======================================
        # NUMÉRICOS E BOOLEANOS permanecem NULL:
        # preencher com 0/False distorce SUM, AVG e taxas
        # (SUM/AVG do Spark já ignoram NULL)
        # ======================================

    return df

In [0]:
def tratar_duplicidades(df, nome_tabela):

    # Duplicidade real = mesma chave primária repetida
    # (linhas inteiras idênticas são impossíveis com PK UUID)
    pk = CHAVES_PRIMARIAS.get(nome_tabela)

    if pk is None or pk not in df.columns:
        return df

    print(f"Tratando duplicidades pela PK ({pk})...")

    antes = df.count()

    df = df.dropDuplicates([pk])

    depois = df.count()

    print(
        f"Linhas removidas: {antes - depois}"
    )

    return df

In [0]:
def tratar_dataframe(df, nome_tabela, catalogo_df):

    print("=" * 70)
    print(f"TRATANDO: {nome_tabela}")
    print("=" * 70)

    # Catálogo somente da tabela atual
    catalogo_tabela = (
        catalogo_df
        .filter(F.col("tabela") == nome_tabela)
    )

    # ==================================================
    # 1 - TRATAR TIPOS
    # ==================================================

    print("\n[1] TIPOS")

    df, alteracoes_tipos = tratar_tipos(
    df,
    nome_tabela,
    tipos_postgres
)

    # ==================================================
    # 2 - TRATAR NULLS
    # ==================================================

    print("\n[2] NULLS")

    tem_nulos = (
        catalogo_tabela
        .filter(F.col("quantidade_nulos") > 0)
        .limit(1)
        .count()
        > 0
    )

    if tem_nulos:

        print("→ Existem NULLs")

        df = tratar_nulos(
            df,
            catalogo_tabela
        )

    else:

        print("→ Nenhum NULL encontrado")

    # ==================================================
    # 3 - TRATAR DUPLICIDADES
    # ==================================================

    print("\n[3] DUPLICIDADES")

    tem_duplicados = (
        catalogo_tabela
        .filter(F.col("quantidade_duplicados") > 0)
        .limit(1)
        .count()
        > 0
    )

    if tem_duplicados:

        print("→ Existem duplicidades")

        df = tratar_duplicidades(
            df,
            nome_tabela
        )

    else:

        print("→ Nenhuma duplicidade encontrada")

    # ==================================================
    # FINAL
    # ==================================================

    print(f"\n✓ {nome_tabela} tratada")

    return df

In [0]:
dfs_tratados = {}

for nome_tabela, df in dfs.items():

    dfs_tratados[nome_tabela] = tratar_dataframe(
        df,
        nome_tabela,
        catalogo_df
    )


TRATANDO: usuarios

[1] TIPOS

Verificando tipos: usuarios
usuario_id: string → string
nome: string → string
email: string → string
cpf: string → string
senha: string → string
senha_hash: string → string
token_redefinicao: string → string
imagem_url: string → string
data_nascimento: date → date
ultimo_acesso: timestamp → timestamp
data_criacao: timestamp → timestamp
data_atualizacao: timestamp → timestamp
data_token_expiracao: timestamp → timestamp
esta_ativo: boolean → boolean

[2] NULLS
→ Existem NULLs
Tratando NULL: senha
Tratando NULL: token_redefinicao
Tratando NULL: imagem_url

[3] DUPLICIDADES
→ Nenhuma duplicidade encontrada

✓ usuarios tratada
TRATANDO: empresas

[1] TIPOS

Verificando tipos: empresas
empresa_id: string → string
nome: string → string
imagem_url: string → string
descricao: string → string

[2] NULLS
→ Existem NULLs
Tratando NULL: imagem_url

[3] DUPLICIDADES
→ Nenhuma duplicidade encontrada

✓ empresas tratada
TRATANDO: empresa_materiais_interesses

[1] TIPOS



In [0]:
# Inspeção: endereços sem CEP
display(dfs_tratados["enderecos"].filter(F.col("cep").isNull()))

endereco_id,cep,logradouro,numero,complemento,bairro,cidade,tipo


##Tratamento específico 

In [0]:
def tratar_especifico_enderecos(df):

    df = (
        df
        .withColumn(
            "cep",
            F.regexp_replace(F.col("cep"), r"\D", "")
        )
        .withColumn(
            "logradouro",
            F.trim(F.col("logradouro"))
        )
        .withColumn(
            "numero",
            F.trim(F.col("numero"))
        )
        .withColumn(
            "bairro",
            F.trim(F.col("bairro"))
        )
        .withColumn(
            "cidade",
            F.initcap(F.trim(F.col("cidade")))
        )
        .withColumn(
            "tipo",
            F.upper(F.trim(F.col("tipo")))
        )
    )

    return df

In [0]:
def tratar_especifico_categorias_materiais(df):

    df = (
        df
        .withColumn(
            "nome_categoria",
            F.initcap(F.trim(F.col("nome_categoria")))
        )
    )

    df = df.filter(
        F.col("nome_categoria").isNotNull() &
        (F.length(F.col("nome_categoria")) > 0)
    )

    return df

In [0]:
def tratar_especifico_materiais(df):

    df = (
        df
        .withColumn(
            "preco_sugerido",
            F.when(
                F.col("preco_sugerido") < 0,
                F.lit(0)
            ).otherwise(F.col("preco_sugerido"))
        )
        .withColumn(
            "esta_disponivel",
            F.coalesce(
                F.col("esta_disponivel"),
                F.lit(True)
            )
        )
    )

    df = df.filter(
        F.col("categoria_id").isNotNull()
    )

    return df

In [0]:
def tratar_especifico_cooperativas(df):

    df = (
        df
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .withColumn(
            "contato_preferencial",
            F.upper(
                F.trim(F.col("contato_preferencial"))
            )
        )
        .withColumn(
            "numero_cooperados",
            F.when(
                F.col("numero_cooperados") < 0,
                F.lit(0)
            ).otherwise(F.col("numero_cooperados"))
        )
    )

    return df

In [0]:
def tratar_especifico_empresas(df):

    df = (
        df
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .withColumn(
            "descricao",
            F.trim(F.col("descricao"))
        )
    )

    return df

In [0]:
def tratar_especifico_empresa_materiais_interesses(df):

    df = df.filter(
        F.col("empresa_id").isNotNull() &
        F.col("categoria_id").isNotNull()
    )

    return df

In [0]:
def tratar_especifico_empresa_cooperativas_favoritas(df):

    df = (
        df
        .filter(
            F.col("empresa_id").isNotNull() &
            F.col("cooperativa_id").isNotNull()
        )
        .withColumn(
            "data_criacao",
            F.to_timestamp("data_criacao")
        )
    )

    return df

In [0]:
def tratar_especifico_perfis(df):

    df = (
        df
        .withColumn(
            "email",
            F.lower(F.trim(F.col("email")))
        )
        .withColumn(
            "cnpj",
            F.regexp_replace(
                F.col("cnpj"),
                r"\D",
                ""
            )
        )
    )

    # Perfil deve pertencer a exatamente um tipo
    df = df.filter(
        (
            F.col("empresa_id").isNotNull() &
            F.col("cooperativa_id").isNull()
        )
        |
        (
            F.col("empresa_id").isNull() &
            F.col("cooperativa_id").isNotNull()
        )
    )

    return df

In [0]:
def tratar_especifico_usuarios(df):

    df = (
        df
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .withColumn(
            "email",
            F.lower(F.trim(F.col("email")))
        )
        .withColumn(
            "cpf",
            F.regexp_replace(
                F.col("cpf"),
                r"\D",
                ""
            )
        )
    )

    # Remove informações sensíveis/desnecessárias para BI
    colunas_remover = [
        "senha",
        "senha_hash",
        "token_redefinicao",
        "data_token_expiracao",
        "cpf"  # dado pessoal desnecessário para BI
    ]

    for coluna in colunas_remover:

        if coluna in df.columns:
            df = df.drop(coluna)

    # Data de nascimento não pode estar no futuro
    df = df.filter(
        F.col("data_nascimento").isNull() |
        (
            F.col("data_nascimento") <
            F.current_date()
        )
    )

    return df

In [0]:
def tratar_especifico_cargos(df):

    df = (
        df
        .withColumn(
            "cargo",
            F.initcap(F.trim(F.col("cargo")))
        )
    )

    return df

In [0]:
def tratar_especifico_status(df):

    df = (
        df
        .withColumn(
            "referencia",
            F.upper(F.trim(F.col("referencia")))
        )
        .withColumn(
            "status_atual",
            F.upper(F.trim(F.col("status_atual")))
        )
    )

    return df

In [0]:
def tratar_especifico_funcionarios(df):

    df = (
        df
        .withColumn(
            "status_funcionario",
            F.upper(
                F.trim(F.col("status_funcionario"))
            )
        )
        .withColumn(
            "esta_ativo",
            F.col("status_funcionario").isin(
                "ATIVO",
                "AFASTADO"
            )
        )
        .withColumn(
            "esta_afastado",
            F.col("status_funcionario") == "AFASTADO"
        )
    )

    return df

In [0]:
def tratar_especifico_telefones(df):

    df = (
        df
        .withColumn(
            "ddd",
            F.regexp_replace(
                F.col("ddd"),
                r"\D",
                ""
            )
        )
        .withColumn(
            "telefone",
            F.regexp_replace(
                F.col("telefone"),
                r"\D",
                ""
            )
        )
    )

    # O telefone deve pertencer a perfil OU usuário
    df = df.filter(
        (
            F.col("perfil_id").isNotNull() &
            F.col("usuario_id").isNull()
        )
        |
        (
            F.col("perfil_id").isNull() &
            F.col("usuario_id").isNotNull()
        )
    )

    return df

In [0]:
def tratar_especifico_equipes(df):

    df = (
        df
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .withColumn(
            "descricao",
            F.trim(F.col("descricao"))
        )
    )

    df = df.filter(
        F.col("gestor_id").isNotNull()
    )

    return df

In [0]:
def tratar_especifico_equipes_cooperados(df):

    return df.filter(
        F.col("equipe_id").isNotNull() &
        F.col("cooperado_id").isNotNull()
    )

In [0]:
def tratar_especifico_pedidos(df):

    df = (
        df
        .withColumn(
            "observacao",
            F.trim(F.col("observacao"))
        )
    )

    # Conclusão não pode ser anterior ao pedido
    df = df.filter(
        F.col("data_conclusao").isNull() |
        (
            F.col("data_conclusao") >=
            F.col("data_pedido")
        )
    )

    # Indicador útil para BI
    df = df.withColumn(
        "pedido_concluido",
        F.col("data_conclusao").isNotNull()
    )

    df = df.withColumn(
        "tempo_conclusao_dias",
        F.when(
            F.col("data_conclusao").isNotNull(),
            F.datediff(
                F.to_date("data_conclusao"),
                F.to_date("data_pedido")
            )
        )
    )

    # Tempo em horas (preserva frações do dia)
    df = df.withColumn(
        "tempo_conclusao_horas",
        F.when(
            F.col("data_conclusao").isNotNull(),
            (F.unix_timestamp("data_conclusao") -
             F.unix_timestamp("data_pedido")) / 3600
        )
    )

    return df

In [0]:
def tratar_especifico_pedido_itens(df):

    df = df.filter(
        F.col("pedido_id").isNotNull() &
        F.col("material_id").isNotNull()
    )

    df = df.filter(
        F.col("quantidade_kg").isNull() |
        (F.col("quantidade_kg") > 0)
    )

    df = df.withColumn(
        "valor_item",
        F.col("quantidade_kg") *
        F.col("preco_unitario")
    )

    return df

In [0]:
def tratar_especifico_pedidos_cooperativas(df):

    return df.filter(
        F.col("pedido_id").isNotNull() &
        F.col("cooperativa_id").isNotNull() &
        F.col("status_id").isNotNull()
    )

In [0]:
def tratar_especifico_negociacoes(df):

    df = df.filter(
        F.col("pedido_id").isNotNull() &
        F.col("cooperativa_id").isNotNull() &
        F.col("empresa_id").isNotNull()
    )

    # Fechamento não pode acontecer antes do início
    df = df.filter(
        F.col("data_fechamento").isNull() |
        (
            F.col("data_fechamento") >=
            F.col("data_inicio")
        )
    )

    # Indicador para BI
    df = df.withColumn(
        "negociacao_fechada",
        F.col("data_fechamento").isNotNull()
    )

    df = df.withColumn(
        "tempo_negociacao_dias",
        F.when(
            F.col("data_fechamento").isNotNull(),
            F.datediff(
                F.to_date("data_fechamento"),
                F.to_date("data_inicio")
            )
        )
    )

    # Tempo em horas (preserva frações do dia)
    df = df.withColumn(
        "tempo_negociacao_horas",
        F.when(
            F.col("data_fechamento").isNotNull(),
            (F.unix_timestamp("data_fechamento") -
             F.unix_timestamp("data_inicio")) / 3600
        )
    )

    return df

In [0]:
def tratar_especifico_negociacao_itens(df):

    df = df.filter(
        F.col("negociacao_id").isNotNull() &
        F.col("material_id").isNotNull()
    )

    df = df.filter(
        F.col("quantidade_kg").isNull() |
        (F.col("quantidade_kg") > 0)
    )

    df = df.withColumn(
        "valor_item",
        F.col("quantidade_kg") *
        F.col("preco_unitario")
    )

    return df

In [0]:
def tratar_especifico_negociacao_mensagens(df):

    df = (
        df
        .withColumn(
            "tipo_mensagem",
            F.upper(
                F.trim(F.col("tipo_mensagem"))
            )
        )
        .withColumn(
            "mensagem",
            F.trim(F.col("mensagem"))
        )
    )

    df = df.filter(
        F.length(F.col("mensagem")) > 0
    )

    return df

In [0]:
def tratar_especifico_avaliacoes(df):

    df = df.filter(
        F.col("avaliador_id") !=
        F.col("avaliado_id")
    )

    df = df.filter(
        F.col("nota").isNull() |
        F.col("nota").between(1, 5)
    )

    # Classificação para os KPIs
    df = df.withColumn(
        "classificacao_nota",
        F.when(F.col("nota") == 5, "Excelente")
        .when(F.col("nota") == 4, "Bom")
        .when(F.col("nota") == 3, "Regular")
        .when(F.col("nota") == 2, "Ruim")
        .when(F.col("nota") == 1, "Muito ruim")
    )

    return df

In [0]:
def tratar_especifico_eventos_operacionais(df):

    df = df.withColumn(
        "data_evento",
        F.to_timestamp("data_evento")
    )

    return df

In [0]:
def tratar_especifico_coletas(df):

    
    df = (
        df
        .withColumn(
            "tipo_coleta",
            F.upper(
                F.trim(F.col("tipo_coleta"))
            )
        )
    )

    # Quantidade coletada deve ser positiva
    df = df.filter(
        F.col("quantidade_kg") > 0
    )

    return df

In [0]:
def tratar_especifico_estoques(df):

    df = df.withColumn(
        "quantidade_kg",
        F.when(
            F.col("quantidade_kg") < 0,
            F.lit(0)
        ).otherwise(
            F.col("quantidade_kg")
        )
    )

    # Indicadores úteis
    df = df.withColumn(
        "estoque_disponivel",
        F.col("quantidade_kg") > 0
    )

    return df

In [0]:
def tratar_especifico_triagens(df):

    df = df.filter(
        F.col("equipe_id").isNotNull() &
        F.col("coleta_id").isNotNull() &
        F.col("material_id").isNotNull()
    )

    df = df.filter(
        F.col("quantidade_kg") > 0
    )

    # Rejeito NULL = 0 (DEFAULT do banco); negativo = 0
    df = df.withColumn(
        "quantidade_rejeito_kg",
        F.greatest(
            F.coalesce(F.col("quantidade_rejeito_kg"), F.lit(0)),
            F.lit(0)
        )
    )

    # Rejeito não pode ser maior que o total
    df = df.filter(
        F.col("quantidade_rejeito_kg") <=
        F.col("quantidade_kg")
    )

    # Quantidade aproveitada
    df = df.withColumn(
        "quantidade_aproveitada_kg",
        F.col("quantidade_kg") -
        F.col("quantidade_rejeito_kg")
    )

    # Taxa por linha (apenas inspeção). Para KPI use SUM/SUM
    # (média de taxas por linha não é ponderada pelo peso)
    df = df.withColumn(
        "taxa_aproveitamento",
        F.when(
            F.col("quantidade_kg") > 0,
            F.col("quantidade_aproveitada_kg") /
            F.col("quantidade_kg")
        )
    )

    return df

In [0]:
def tratar_especifico_movimentacoes_estoques(df):

    df = df.filter(
        F.col("quantidade_kg") != 0
    )

    # Tipo da movimentação
    df = df.withColumn(
        "tipo_movimentacao",
        F.when(
            F.col("quantidade_kg") > 0,
            F.lit("ENTRADA")
        ).otherwise(
            F.lit("SAIDA")
        )
    )

    # Quantidade absoluta movimentada
    df = df.withColumn(
        "quantidade_movimentada_kg",
        F.abs(F.col("quantidade_kg"))
    )

    # Identifica a origem
    df = df.withColumn(
        "origem_movimentacao",
        F.when(
            F.col("triagem_id").isNotNull(),
            F.lit("TRIAGEM")
        )
        .when(
            F.col("item_id").isNotNull(),
            F.lit("PEDIDO")
        )
    )

    return df

In [0]:
def tratar_especifico_tipos_rateios(df):

    df = (
        df
        .withColumn(
            "tipo_rateio",
            F.trim(F.col("tipo_rateio"))
        )
        .withColumn(
            "descricao",
            F.trim(F.col("descricao"))
        )
    )

    return df

In [0]:
def tratar_especifico_rateios(df):

    df = df.withColumn(
        "mes_referencia",
        F.to_date("mes_referencia")
    )

    # Padronizar para o primeiro dia do mês
    df = df.withColumn(
        "mes_referencia",
        F.trunc(
            F.col("mes_referencia"),
            "month"
        )
    )

    # Indicador útil
    df = df.withColumn(
        "ano",
        F.year("mes_referencia")
    )

    df = df.withColumn(
        "mes",
        F.month("mes_referencia")
    )

    return df

In [0]:
def tratar_especifico_rateios_funcionarios(df):

    df = df.filter(
        F.col("rateio_id").isNotNull() &
        F.col("cooperado_id").isNotNull()
    )

    df = df.filter(
        F.col("valor_rateio") > 0
    )

    return df

In [0]:
def tratar_especifico_despesas(df):

    df = (
        df
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .withColumn(
            "tipo_despesa",
            F.upper(
                F.trim(F.col("tipo_despesa"))
            )
        )
    )

    return df

In [0]:
def tratar_especifico_lancamentos_despesas(df):

    df = (
        df
        .withColumn(
            "mes_referencia",
            F.trunc(
                F.to_date("mes_referencia"),
                "month"
            )
        )
        .withColumn(
            "valor",
            F.when(
                F.col("valor") < 0,
                F.lit(0)
            ).otherwise(F.col("valor"))
        )
    )

    df = df.withColumn(
        "ano",
        F.year("mes_referencia")
    )

    df = df.withColumn(
        "mes",
        F.month("mes_referencia")
    )

    return df

In [0]:
def tratar_especifico_rotas(df):

    df = (
        df
        .withColumn(
            "nome",
            F.trim(F.col("nome"))
        )
        .withColumn(
            "esta_ativa",
            F.coalesce(
                F.col("esta_ativa"),
                F.lit(True)
            )
        )
    )

    return df

In [0]:
def tratar_especifico_rotas_enderecos(df):

    df = (
        df
        .withColumn(
            "nome_local",
            F.trim(F.col("nome_local"))
        )
        .withColumn(
            "tipo_local",
            F.upper(
                F.trim(F.col("tipo_local"))
            )
        )
    )

    # Ordem deve ser positiva
    df = df.filter(
        F.col("ordem") > 0
    )

    return df

In [0]:
tratamentos_especificos = {

    "enderecos":
        tratar_especifico_enderecos,

    "categorias_materiais":
        tratar_especifico_categorias_materiais,

    "materiais":
        tratar_especifico_materiais,

    "cooperativas":
        tratar_especifico_cooperativas,

    "empresas":
        tratar_especifico_empresas,

    "empresa_materiais_interesses":
        tratar_especifico_empresa_materiais_interesses,

    "empresa_cooperativas_favoritas":
        tratar_especifico_empresa_cooperativas_favoritas,

    "perfis":
        tratar_especifico_perfis,

    "usuarios":
        tratar_especifico_usuarios,

    "cargos":
        tratar_especifico_cargos,

    "status":
        tratar_especifico_status,

    "funcionarios":
        tratar_especifico_funcionarios,

    "telefones":
        tratar_especifico_telefones,

    "equipes":
        tratar_especifico_equipes,

    "equipes_cooperados":
        tratar_especifico_equipes_cooperados,

    "pedidos":
        tratar_especifico_pedidos,

    "pedido_itens":
        tratar_especifico_pedido_itens,

    "pedidos_cooperativas":
        tratar_especifico_pedidos_cooperativas,

    "negociacoes":
        tratar_especifico_negociacoes,

    "negociacao_itens":
        tratar_especifico_negociacao_itens,

    "negociacao_mensagens":
        tratar_especifico_negociacao_mensagens,

    "avaliacoes":
        tratar_especifico_avaliacoes,

    "eventos_operacionais":
        tratar_especifico_eventos_operacionais,

    "coletas":
        tratar_especifico_coletas,

    "estoques":
        tratar_especifico_estoques,

    "triagens":
        tratar_especifico_triagens,

    "movimentacoes_estoques":
        tratar_especifico_movimentacoes_estoques,

    "tipos_rateios":
        tratar_especifico_tipos_rateios,

    "rateios":
        tratar_especifico_rateios,

    "rateios_funcionarios":
        tratar_especifico_rateios_funcionarios,

    "despesas":
        tratar_especifico_despesas,

    "lancamentos_despesas":
        tratar_especifico_lancamentos_despesas,

    "rotas":
        tratar_especifico_rotas,

    "rotas_enderecos":
        tratar_especifico_rotas_enderecos
}

In [0]:
def aplicar_tratamento_especifico(
    df,
    nome_tabela
):

    funcao = tratamentos_especificos.get(
        nome_tabela
    )

    if funcao is None:

        print(
            f"⚠️ Nenhum tratamento específico "
            f"para {nome_tabela}"
        )

        return df

    print(
        f"→ Aplicando tratamento específico: "
        f"{nome_tabela}"
    )

    return funcao(df)

In [0]:
dfs_especificos = {}

for nome_tabela, df in dfs_tratados.items():

    print("=" * 60)
    print(f"TRATAMENTO ESPECÍFICO: {nome_tabela}")
    print("=" * 60)

    dfs_especificos[nome_tabela] = aplicar_tratamento_especifico(
        df,
        nome_tabela
    )

TRATAMENTO ESPECÍFICO: usuarios
→ Aplicando tratamento específico: usuarios
TRATAMENTO ESPECÍFICO: empresas
→ Aplicando tratamento específico: empresas
TRATAMENTO ESPECÍFICO: empresa_materiais_interesses
→ Aplicando tratamento específico: empresa_materiais_interesses
TRATAMENTO ESPECÍFICO: empresa_cooperativas_favoritas
→ Aplicando tratamento específico: empresa_cooperativas_favoritas
TRATAMENTO ESPECÍFICO: cargos
→ Aplicando tratamento específico: cargos
TRATAMENTO ESPECÍFICO: status
→ Aplicando tratamento específico: status
TRATAMENTO ESPECÍFICO: materiais
→ Aplicando tratamento específico: materiais
TRATAMENTO ESPECÍFICO: perfis
→ Aplicando tratamento específico: perfis
TRATAMENTO ESPECÍFICO: enderecos
→ Aplicando tratamento específico: enderecos
TRATAMENTO ESPECÍFICO: telefones
→ Aplicando tratamento específico: telefones
TRATAMENTO ESPECÍFICO: equipes
→ Aplicando tratamento específico: equipes
TRATAMENTO ESPECÍFICO: equipes_cooperados
→ Aplicando tratamento específico: equipes_coo

In [0]:
# Catálogo após os tratamentos (mesma função da célula "Criação do catálogo")
catalogo_final_df = gerar_catalogo(dfs_especificos)

display(catalogo_final_df)

Analisando usuarios...
Analisando empresas...
Analisando empresa_materiais_interesses...
Analisando empresa_cooperativas_favoritas...
Analisando cargos...
Analisando status...
Analisando materiais...
Analisando perfis...
Analisando enderecos...
Analisando telefones...
Analisando equipes...
Analisando equipes_cooperados...
Analisando funcionarios...
Analisando pedidos_cooperativas...
Analisando pedido_itens...
Analisando negociacoes...
Analisando negociacao_mensagens...
Analisando avaliacoes...
Analisando eventos_operacionais...
Analisando coletas...
Analisando tipos_rateios...
Analisando estoques...
Analisando triagens...
Analisando movimentacoes_estoques...
Analisando negociacao_itens...
Analisando lancamentos_despesas...
Analisando rotas_enderecos...
Analisando rotas...
Analisando rateios_funcionarios...
Analisando despesas...
Analisando rateios...
Analisando categorias_materiais...
Analisando cooperativas...
Analisando pedidos...
Analisando rota_execucoes...
Analisando rota_execucoe

tabela,coluna,tipo,quantidade_linhas,quantidade_colunas,quantidade_nulos,percentual_nulos,valores_distintos,quantidade_duplicados,percentual_duplicados,minimo,maximo,media,desvio_padrao
usuarios,usuario_id,string,416,9,0,0.0,416,0,0.0,null,null,null,null
usuarios,nome,string,416,9,0,0.0,415,0,0.0,null,null,null,null
usuarios,email,string,416,9,0,0.0,416,0,0.0,null,null,null,null
usuarios,imagem_url,string,416,9,0,0.0,3,0,0.0,null,null,null,null
usuarios,data_nascimento,date,416,9,9,2.1634615384615383,402,0,0.0,null,null,null,null
usuarios,ultimo_acesso,timestamp,416,9,398,95.67307692307693,18,0,0.0,null,null,null,null
usuarios,data_criacao,timestamp,416,9,1,0.2403846153846154,6,0,0.0,null,null,null,null
usuarios,data_atualizacao,timestamp,416,9,1,0.2403846153846154,6,0,0.0,null,null,null,null
usuarios,esta_ativo,boolean,416,9,1,0.2403846153846154,2,0,0.0,null,null,null,null
empresas,empresa_id,string,49,4,0,0.0,49,0,0.0,null,null,null,null


In [0]:
print("Quantidade de tabelas:")
print(len(dfs_especificos))

print("\nTabelas:")
print(list(dfs_especificos.keys()))

Quantidade de tabelas:
36

Tabelas:
['usuarios', 'empresas', 'empresa_materiais_interesses', 'empresa_cooperativas_favoritas', 'cargos', 'status', 'materiais', 'perfis', 'enderecos', 'telefones', 'equipes', 'equipes_cooperados', 'funcionarios', 'pedidos_cooperativas', 'pedido_itens', 'negociacoes', 'negociacao_mensagens', 'avaliacoes', 'eventos_operacionais', 'coletas', 'tipos_rateios', 'estoques', 'triagens', 'movimentacoes_estoques', 'negociacao_itens', 'lancamentos_despesas', 'rotas_enderecos', 'rotas', 'rateios_funcionarios', 'despesas', 'rateios', 'categorias_materiais', 'cooperativas', 'pedidos', 'rota_execucoes', 'rota_execucoes_paradas']


##Criação de indíces (colunas)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Fonte: saída do loop de tratamentos específicos (célula 56 do Notebook)
base = dfs_especificos

# ------------------------------------------------------------------
# Domínio de status: o tables.sql define a tabela `status`
# (referencia, status_atual), mas NÃO define os valores possíveis.
# Preencha as listas com os valores reais após executar:
#     display(base["status"])  (célula "Domínio de status" abaixo)
# Enquanto uma lista estiver vazia, os KPIs que dependem dela
# ficam NULL (lacuna explícita) em vez de receberem valor inventado.
# ------------------------------------------------------------------
STATUS_NEGOCIACAO_SUCESSO = ["ACORDO FECHADO"]            # referencia NEGOCIACAO (valores da célula "Domínio de status")
STATUS_PEDIDO_COOP_CANCELADO = ["RECUSADO", "CANCELADO"]   # PEDIDO/RECUSADO e GERAL/CANCELADO


def flag_status(col_status, valores):
    """Boolean de status; NULL quando a lista ainda não foi configurada."""
    if not valores:
        return F.lit(None).cast("boolean")
    return F.upper(F.trim(F.col(col_status))).isin([v.upper() for v in valores])


def mes(col):
    """Primeiro dia do mês (mesmo padrão de mes_referencia em rateios)."""
    return F.trunc(F.col(col), "month")


def horas_entre(inicio, fim):
    """Diferença em horas (double), preservando frações do dia."""
    return (F.unix_timestamp(F.col(fim)) - F.unix_timestamp(F.col(inicio))) / 3600.0


def razao(num, den, pct=False):
    """Divisão segura: NULL quando o denominador é 0/NULL (não 0)."""
    fator = F.lit(100.0) if pct else F.lit(1.0)
    return F.when(F.col(den) > 0, F.col(num) / F.col(den) * fator)


def variacao_pct(col, col_anterior):
    return F.when(
        F.col(col_anterior) > 0,
        (F.col(col) - F.col(col_anterior)) / F.col(col_anterior) * 100.0,
    )


df_status = base["status"].select("status_id", "referencia", "status_atual")

In [0]:
# Domínio de status: valores usados para preencher STATUS_* na célula de parâmetros
display(
    base["status"]
    .select("referencia", "status_atual")
    .distinct()
    .orderBy("referencia", "status_atual")
)

referencia,status_atual
COLETA,AGENDADA
COLETA,COLETADO
COLETA,PENDENTE
GERAL,CANCELADO
NEGOCIACAO,ACORDO FECHADO
NEGOCIACAO,EM ANDAMENTO
NEGOCIACAO,NEGOCIAÇÃO RECUSADA
PEDIDO,ABERTO
PEDIDO,ACEITO
PEDIDO,EM NEGOCIAÇÃO


In [0]:
# negociacao_itens e negociacao_mensagens são 1:N -> pré-agregados por
# negociacao_id ANTES do join (evita multiplicar valor_total).
itens_neg_agg = (
    base["negociacao_itens"]
    .groupBy("negociacao_id")
    .agg(
        F.sum("quantidade_kg").alias("kg_negociado"),
        F.sum(F.col("quantidade_kg") * F.col("preco_unitario")).alias("valor_itens_negociado"),
    )
)

msgs_agg = (
    base["negociacao_mensagens"]
    .groupBy("negociacao_id")
    .agg(
        F.sum(F.when(F.col("tipo_mensagem") == "CONTRAPROPOSTA", 1).otherwise(0))
         .alias("qtd_contrapropostas"),
    )
)

df_negociacoes_base = (
    base["negociacoes"]
    .join(F.broadcast(df_status), "status_id", "left")          # N:1
    .join(itens_neg_agg, "negociacao_id", "left")                # 1:1 após agg
    .join(msgs_agg, "negociacao_id", "left")                     # 1:1 após agg
    .withColumn("valor_negociacao",
                F.coalesce(F.col("valor_total"), F.col("valor_itens_negociado")))
    .withColumn("negociacao_sucesso", flag_status("status_atual", STATUS_NEGOCIACAO_SUCESSO))
    .withColumn("negociacao_encerrada", F.col("data_fechamento").isNotNull())
    .withColumn("tempo_negociacao_horas",
                F.when(F.col("data_fechamento").isNotNull(),
                       horas_entre("data_inicio", "data_fechamento")))
    .withColumn("qtd_contrapropostas", F.coalesce("qtd_contrapropostas", F.lit(0)))
    .withColumn("mes_inicio", mes("data_inicio"))
    .withColumn("mes_fechamento", mes("data_fechamento"))
)

In [0]:
itens_ped_agg = (
    base["pedido_itens"]
    .groupBy("pedido_id")
    .agg(
        F.sum("quantidade_kg").alias("kg_pedido"),
    )
)

neg_por_pedido = (
    df_negociacoes_base
    .groupBy("pedido_id")
    .agg(
        F.count("*").alias("qtd_negociacoes"),
        F.sum(F.col("negociacao_sucesso").cast("int")).alias("qtd_negociacoes_sucesso"),
        F.sum(F.when(F.col("negociacao_sucesso"), F.col("kg_negociado"))).alias("kg_negociado_sucesso"),
    )
)

df_pedidos_base = (
    base["pedidos"]
    .join(itens_ped_agg, "pedido_id", "left")                    # 1:1 após agg
    .join(neg_por_pedido, "pedido_id", "left")                   # 1:1 após agg
    .withColumn("mes_pedido", mes("data_pedido"))
    .withColumn("pedido_concluido", F.col("data_conclusao").isNotNull())
    .withColumn("tempo_conclusao_horas",
                F.when(F.col("data_conclusao").isNotNull(),
                       horas_entre("data_pedido", "data_conclusao")))
    .withColumn("pedido_convertido",
                F.when(F.lit(bool(STATUS_NEGOCIACAO_SUCESSO)),
                       F.coalesce("qtd_negociacoes_sucesso", F.lit(0)) > 0))
)

In [0]:
# Início: primeiro mês com pedido OU coleta; fim: mês corrente.
limites = (
    df_pedidos_base.select(F.col("mes_pedido").alias("m"))
    .unionByName(base["coletas"].select(mes("data_evento").alias("m")))
    .agg(F.min("m").alias("ini"))
    .withColumn("fim", F.trunc(F.current_date(), "month"))
)
df_calendario = limites.select(
    F.explode(F.sequence("ini", "fim", F.expr("interval 1 month"))).alias("mes_referencia")
)

##Criação/cálculos dos KPIs

###Estratégicos

In [0]:
w_mes = Window.orderBy("mes_referencia")   # poucas linhas (1 por mês): OK

ped_mes = (
    df_pedidos_base
    .groupBy(F.col("mes_pedido").alias("mes_referencia"))
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum(F.col("pedido_concluido").cast("int")).alias("pedidos_concluidos"),
        F.avg("tempo_conclusao_horas").alias("tempo_medio_conclusao_horas"),
        F.sum(F.col("pedido_convertido").cast("int")).alias("pedidos_convertidos"),
        F.countDistinct("empresa_id").alias("empresas_ativas"),
        F.sum("kg_pedido").alias("kg_pedido_mes"),
        F.sum("kg_negociado_sucesso").alias("kg_atendido_mes"),
    )
)

neg_fech_mes = (
    df_negociacoes_base
    .filter(F.col("negociacao_sucesso"))
    .groupBy(F.col("mes_fechamento").alias("mes_referencia"))
    .agg(
        F.sum("valor_negociacao").alias("valor_negociado_total"),
        F.count("*").alias("qtd_negociacoes_sucesso"),
    )
)

coop_ativas_mes = (
    df_negociacoes_base
    .groupBy(F.col("mes_inicio").alias("mes_referencia"))
    .agg(F.countDistinct("cooperativa_id").alias("cooperativas_ativas"))
)

emp_mes = (df_pedidos_base
           .select("empresa_id", F.col("mes_pedido").alias("mes_referencia"))
           .distinct())
retencao_mes = (
    emp_mes.alias("a")
    .join(emp_mes.alias("b"),
          (F.col("a.empresa_id") == F.col("b.empresa_id")) &
          (F.col("b.mes_referencia") == F.add_months(F.col("a.mes_referencia"), 1)),
          "left")
    .groupBy(F.add_months(F.col("a.mes_referencia"), 1).alias("mes_referencia"))
    .agg(F.count("*").alias("empresas_base_mes_anterior"),
         F.count(F.col("b.empresa_id")).alias("empresas_retidas"))
)

aval_mes = (
    base["avaliacoes"]
    .filter(F.col("nota").between(1, 5))
    .groupBy(mes("data_avaliacao").alias("mes_referencia"))
    .agg(F.avg("nota").alias("nota_media_avaliacao"))
)

df_kpi_plataforma_mes = (
    df_calendario
    .join(ped_mes, "mes_referencia", "left")
    .join(neg_fech_mes, "mes_referencia", "left")
    .join(coop_ativas_mes, "mes_referencia", "left")
    .join(retencao_mes, "mes_referencia", "left")
    .join(aval_mes, "mes_referencia", "left")
    .fillna(0, subset=["qtd_pedidos", "pedidos_concluidos", "empresas_ativas",
                       "cooperativas_ativas", "qtd_negociacoes_sucesso"])
    # ---------- ESTRATÉGICOS ----------
    .withColumn("valor_negociado_total",
                F.when(F.lit(bool(STATUS_NEGOCIACAO_SUCESSO)),
                       F.coalesce("valor_negociado_total", F.lit(0))))
    .withColumn("valor_negociado_ant", F.lag("valor_negociado_total").over(w_mes))
    .withColumn("crescimento_valor_negociado_pct",
                variacao_pct("valor_negociado_total", "valor_negociado_ant"))
    .withColumn("ticket_medio_negociacao",
                razao("valor_negociado_total", "qtd_negociacoes_sucesso"))
    .withColumn("qtd_pedidos_ant", F.lag("qtd_pedidos").over(w_mes))
    .withColumn("crescimento_pedidos_pct", variacao_pct("qtd_pedidos", "qtd_pedidos_ant"))
    .withColumn("taxa_retencao_empresas_pct",
                razao("empresas_retidas", "empresas_base_mes_anterior", pct=True))
    .withColumn("taxa_conversao_pedidos_pct",
                razao("pedidos_convertidos", "qtd_pedidos", pct=True))
    # ---------- OPERACIONAIS ----------
    .withColumn("taxa_conclusao_pedidos_pct",
                razao("pedidos_concluidos", "qtd_pedidos", pct=True))
    .withColumn("taxa_atendimento_volume_pct",
                razao("kg_atendido_mes", "kg_pedido_mes", pct=True))
    .drop("valor_negociado_ant", "qtd_pedidos_ant")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(df_kpi_plataforma_mes)

mes_referencia,qtd_pedidos,pedidos_concluidos,tempo_medio_conclusao_horas,pedidos_convertidos,empresas_ativas,kg_pedido_mes,kg_atendido_mes,valor_negociado_total,qtd_negociacoes_sucesso,cooperativas_ativas,empresas_base_mes_anterior,empresas_retidas,nota_media_avaliacao,crescimento_valor_negociado_pct,ticket_medio_negociacao,crescimento_pedidos_pct,taxa_retencao_empresas_pct,taxa_conversao_pedidos_pct,taxa_conclusao_pedidos_pct,taxa_atendimento_volume_pct
2026-04-01,13,9,93.33333333333333,1,11,9695.445,536.550,0.00000,0,2,null,null,null,null,null,null,null,7.6923076923076925,69.23076923076923,5.534
2026-05-01,59,49,92.08163265306122,9,35,68388.828,4375.925,0.00000,0,8,11,10,null,null,null,353.8461538461538,90.9090909090909,15.254237288135593,83.05084745762711,6.3986
2026-06-01,59,43,109.3953488372093,12,34,39817.470,7659.827,0.00000,0,14,35,26,4.0,null,null,0.0,74.28571428571429,20.33898305084746,72.88135593220339,19.237399999999997
2026-07-01,50,43,108.83720930232558,13,30,51923.252,5587.566,0.00000,0,11,34,25,4.127450980392157,null,null,-15.254237288135593,73.52941176470588,26.0,86.0,10.7612
2026-08-01,47,39,88.28205128205128,8,32,22100.664,4199.840,0.00000,0,8,30,17,4.29,null,null,-6.0,56.666666666666664,17.02127659574468,82.97872340425532,19.0032
2026-09-01,36,26,97.84615384615384,7,26,31189.138,4342.193,74273.51000,50,10,32,16,4.041666666666667,null,1485.4702,-23.404255319148938,50.0,19.444444444444446,72.22222222222221,13.9221
2026-10-01,4,0,null,0,3,3040.000,null,0.00000,0,2,26,2,null,-100.0,null,-88.88888888888889,7.6923076923076925,0.0,0.0,null


###Operacionais 

In [0]:

CHAVE_COOPERADO = "funcionario_id"   # ou "usuario_id"

w_func = Window.partitionBy(CHAVE_COOPERADO).orderBy(F.col("data_admissao").desc())
dim_cooperado = (
    base["funcionarios"]
    .filter(F.col("cooperativa_id").isNotNull())
    # garante 1 linha por chave (necessário se a chave for usuario_id: 1 usuário
    # pode ter N vínculos em funcionarios -> join N:N)
    .withColumn("_rn", F.row_number().over(w_func))
    .filter(F.col("_rn") == 1)
    .select(F.col(CHAVE_COOPERADO).alias("cooperado_id"), "cooperativa_id")
)

cooperados_ativos_mes = (
    base["funcionarios"]
    .filter(F.col("cooperativa_id").isNotNull())
    .crossJoin(F.broadcast(df_calendario))   # funcionários x meses (tabelas pequenas)
    .filter(F.col("data_admissao") < F.add_months(F.col("mes_referencia"), 1))
    .filter(F.col("data_desligamento").isNull() |
            (F.col("data_desligamento") >= F.col("mes_referencia").cast("timestamp")))
    .groupBy("cooperativa_id", "mes_referencia")
    .agg(F.countDistinct("funcionario_id").alias("cooperados_ativos"))
)

In [0]:
chave = ["cooperativa_id", "mes_referencia"]

coletas_coop = (
    base["coletas"]
    .join(dim_cooperado, "cooperado_id", "left")                              # N:1
    .join(base["rotas"].select("rota_id", F.col("cooperativa_id").alias("coop_rota")),
          "rota_id", "left")                                                  # N:1
    .withColumn("cooperativa_id", F.coalesce("cooperativa_id", "coop_rota"))
    .withColumn("mes_referencia", mes("data_evento"))
)
coletas_mes = (
    coletas_coop.filter(F.col("cooperativa_id").isNotNull())
    .groupBy(*chave)
    .agg(F.count("*").alias("qtd_coletas"),
         F.sum("quantidade_kg").alias("volume_coletado_kg"))
)

triagens_coop = (
    base["triagens"]
    .join(coletas_coop.select(F.col("evento_id").alias("coleta_id"),
                              "cooperativa_id",
                              F.col("data_evento").alias("data_coleta")),
          "coleta_id", "left")                                                # N:1
    .withColumn("rejeito_kg", F.coalesce("quantidade_rejeito_kg", F.lit(0)))
    .withColumn("mes_referencia", mes("data_evento"))
)
triagens_mes = (
    triagens_coop.filter(F.col("cooperativa_id").isNotNull())
    .groupBy(*chave)
    .agg(F.sum("quantidade_kg").alias("volume_triado_kg"),
         F.sum("rejeito_kg").alias("volume_rejeito_kg"),
         F.avg(horas_entre("data_coleta", "data_evento")).alias("tempo_medio_coleta_triagem_horas"))
    .withColumn("volume_aproveitado_kg", F.col("volume_triado_kg") - F.col("volume_rejeito_kg"))
)

expedicao_mes = (
    base["movimentacoes_estoques"]
    .filter((F.col("quantidade_kg") < 0) & F.col("item_id").isNotNull())      # SAÍDA por pedido
    .join(base["estoques"].select("estoque_id", "cooperativa_id"), "estoque_id", "inner")  # N:1
    .groupBy("cooperativa_id", mes("data_movimentacao").alias("mes_referencia"))
    .agg(F.sum(F.abs("quantidade_kg")).alias("volume_expedido_kg"))
)

neg_coop_inicio = (
    df_negociacoes_base
    .join(base["pedidos"].select("pedido_id", "data_pedido"), "pedido_id", "left")  # N:1
    .groupBy("cooperativa_id", F.col("mes_inicio").alias("mes_referencia"))
    .agg(F.count("*").alias("negociacoes_iniciadas"),
         F.avg(horas_entre("data_pedido", "data_inicio")).alias("tempo_medio_ate_negociacao_horas"))
)

neg_coop_fech = (
    df_negociacoes_base
    .filter(F.col("negociacao_encerrada"))
    .groupBy("cooperativa_id", F.col("mes_fechamento").alias("mes_referencia"))
    .agg(F.count("*").alias("negociacoes_encerradas"),
         F.sum(F.col("negociacao_sucesso").cast("int")).alias("negociacoes_sucesso"),
         F.sum(F.when(F.col("negociacao_sucesso"), F.col("valor_negociacao"))).alias("valor_negociado"),
         F.avg("tempo_negociacao_horas").alias("tempo_medio_negociacao_horas"),
         F.avg("qtd_contrapropostas").alias("media_contrapropostas_negociacao"))
)

direcionamentos_mes = (
    base["pedidos_cooperativas"]
    .join(F.broadcast(df_status), "status_id", "left")                         # N:1
    .join(base["pedidos"].select("pedido_id", "data_pedido"), "pedido_id", "inner")  # N:1
    .withColumn("cancelado", flag_status("status_atual", STATUS_PEDIDO_COOP_CANCELADO))
    .groupBy("cooperativa_id", mes("data_pedido").alias("mes_referencia"))
    .agg(F.count("*").alias("pedidos_direcionados"),
         F.sum(F.col("cancelado").cast("int")).alias("pedidos_cancelados"))
)

despesas_mes = (
    base["lancamentos_despesas"]
    .join(base["despesas"].select("despesa_id", "cooperativa_id"), "despesa_id", "inner")  # N:1
    .groupBy("cooperativa_id", F.trunc("mes_referencia", "month").alias("mes_referencia"))
    .agg(F.sum("valor").alias("despesa_total"))
)

rateio_mes = (
    base["rateios"]
    .join(base["rateios_funcionarios"], "rateio_id", "inner")                   # 1:N (intencional)
    .groupBy("cooperativa_id", F.trunc("mes_referencia", "month").alias("mes_referencia"))
    .agg(F.sum("valor_rateio").alias("valor_rateio_total"),
         F.countDistinct("cooperado_id").alias("cooperados_rateio"))
)

parciais = [coletas_mes, triagens_mes, expedicao_mes, neg_coop_inicio, neg_coop_fech,
            direcionamentos_mes, despesas_mes, rateio_mes, cooperados_ativos_mes]

In [0]:
grade = parciais[0].select(*chave)
for p in parciais[1:]:
    grade = grade.unionByName(p.select(*chave))
grade = grade.distinct()

df_kpi_cooperativa_mes = grade
for p in parciais:                     # cada parcial é único na chave -> join 1:1
    df_kpi_cooperativa_mes = df_kpi_cooperativa_mes.join(p, chave, "left")

contagens = ["qtd_coletas", "volume_coletado_kg", "volume_triado_kg", "volume_rejeito_kg",
             "volume_aproveitado_kg", "volume_expedido_kg", "negociacoes_iniciadas",
             "negociacoes_encerradas", "pedidos_direcionados", "despesa_total",
             "cooperados_ativos"]
w_coop_mes = Window.partitionBy("mes_referencia")

df_kpi_cooperativa_mes = (
    df_kpi_cooperativa_mes
    .fillna(0, subset=contagens)
    .withColumn("valor_negociado",
                F.when(F.lit(bool(STATUS_NEGOCIACAO_SUCESSO)),
                       F.coalesce("valor_negociado", F.lit(0))))
    # ---------- ESTRATÉGICOS ----------
    .withColumn("participacao_valor_negociado_pct",
                F.when(F.sum("valor_negociado").over(w_coop_mes) > 0,
                       F.col("valor_negociado") / F.sum("valor_negociado").over(w_coop_mes) * 100))
    .withColumn("margem_operacional_pct",
                F.when(F.col("valor_negociado") > 0,
                       (F.col("valor_negociado") - F.col("despesa_total"))
                       / F.col("valor_negociado") * 100))
    .withColumn("valor_medio_rateio_cooperado", razao("valor_rateio_total", "cooperados_rateio"))
    # ---------- OPERACIONAIS ----------
    .withColumn("taxa_cancelamento_pct", razao("pedidos_cancelados", "pedidos_direcionados", pct=True))
    .withColumn("taxa_fechamento_negociacoes_pct",
                razao("negociacoes_sucesso", "negociacoes_encerradas", pct=True))
    .withColumn("taxa_rejeito_triagem_pct", razao("volume_rejeito_kg", "volume_triado_kg", pct=True))
    # tempo_medio_negociacao_horas, media_contrapropostas_negociacao,
    # tempo_medio_ate_negociacao_horas, volume_coletado_kg, volume_expedido_kg
    # já são colunas de KPI vindas das parciais.
    # ---------- PRODUTIVIDADE ----------
    .withColumn("kg_coletado_por_cooperado", razao("volume_coletado_kg", "cooperados_ativos"))
    .withColumn("receita_por_cooperado", razao("valor_negociado", "cooperados_ativos"))
    .withColumn("custo_por_kg_aproveitado", razao("despesa_total", "volume_aproveitado_kg"))
    .withColumn("kg_medio_por_coleta", razao("volume_coletado_kg", "qtd_coletas"))
    # tempo_medio_coleta_triagem_horas já vem de triagens_mes
)

In [0]:
display(df_kpi_cooperativa_mes)

cooperativa_id,mes_referencia,qtd_coletas,volume_coletado_kg,volume_triado_kg,volume_rejeito_kg,tempo_medio_coleta_triagem_horas,volume_aproveitado_kg,volume_expedido_kg,negociacoes_iniciadas,tempo_medio_ate_negociacao_horas,negociacoes_encerradas,negociacoes_sucesso,valor_negociado,tempo_medio_negociacao_horas,media_contrapropostas_negociacao,pedidos_direcionados,pedidos_cancelados,despesa_total,valor_rateio_total,cooperados_rateio,cooperados_ativos,participacao_valor_negociado_pct,margem_operacional_pct,valor_medio_rateio_cooperado,taxa_cancelamento_pct,taxa_fechamento_negociacoes_pct,taxa_rejeito_triagem_pct,kg_coletado_por_cooperado,receita_por_cooperado,custo_por_kg_aproveitado,kg_medio_por_coleta
2415a263-6475-449d-858c-49135f5ab0ac,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,2,24.0,0,null,0.00000,null,null,3,2,0.00,null,null,20,null,null,null,66.66666666666666,null,null,0.0,0.0,null,null
5a83c19a-84b6-410f-a565-d74a1bb8f537,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,1,24.0,0,null,0.00000,null,null,2,1,0.00,null,null,21,null,null,null,50.0,null,null,0.0,0.0,null,null
ca5c9df1-3567-4a59-8952-d2aec372909a,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,1,1,0.00,null,null,20,null,null,null,100.0,null,null,0.0,0.0,null,null
ab87b483-01b2-4db3-8c4f-b67085bfa22d,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,4,0,0.00,null,null,19,null,null,null,0.0,null,null,0.0,0.0,null,null
e3e8fcd7-2052-460e-8671-9712d19e315e,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,1,0,0.00,null,null,18,null,null,null,0.0,null,null,0.0,0.0,null,null
906ecbcf-9b49-4293-909c-a03e5e7a9d3c,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,2,0,0.00,null,null,21,null,null,null,0.0,null,null,0.0,0.0,null,null
a837f9cd-61a1-4152-815e-47b34b956480,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,1,0,0.00,null,null,21,null,null,null,0.0,null,null,0.0,0.0,null,null
6fc17b82-1e3d-438b-bd01-d2286fc3c5e4,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,1,0,0.00,null,null,18,null,null,null,0.0,null,null,0.0,0.0,null,null
416ea17e-96e6-4cad-bbf1-9babe6822dc3,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,2,1,0.00,null,null,19,null,null,null,50.0,null,null,0.0,0.0,null,null
b77534ad-0b16-46d8-8e74-0cab423fa243,2026-04-01,0,0.000,0.000,0.000,null,0.000,0.000,0,null,0,null,0.00000,null,null,1,0,0.00,null,null,20,null,null,null,0.0,null,null,0.0,0.0,null,null


In [0]:
JANELA_DIAS = 90
saidas_janela = (
    base["movimentacoes_estoques"]
    .filter((F.col("quantidade_kg") < 0) &
            (F.col("data_movimentacao") >= F.date_sub(F.current_date(), JANELA_DIAS)))
    .groupBy("estoque_id")
    .agg(F.sum(F.abs("quantidade_kg")).alias("saida_kg_janela"))
)

df_kpi_estoque = (
    base["estoques"]
    .join(saidas_janela, "estoque_id", "left")                                # 1:1 após agg
    .withColumn("saida_media_diaria_kg",
                F.coalesce("saida_kg_janela", F.lit(0)) / F.lit(JANELA_DIAS))
    .withColumn("cobertura_estoque_dias",
                razao("quantidade_kg", "saida_media_diaria_kg"))
)

In [0]:
display(df_kpi_estoque)

estoque_id,cooperativa_id,material_id,quantidade_kg,data_atualizacao,estoque_disponivel,saida_kg_janela,saida_media_diaria_kg,cobertura_estoque_dias
3965c94b-2717-4955-8bcc-5c38b721c7bf,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,9bcb2a9c-afed-47db-9700-590ec2d036de,4286.159,2026-09-21T12:33:34.843Z,true,null,0.000000,null
e88db7c9-96cd-4e52-a2fd-33bda6e5dc5a,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,95d462db-6add-4837-9808-19ef70a68f65,4663.834,2026-09-21T12:33:34.843Z,true,null,0.000000,null
d6eea141-86fe-4984-a0ff-97652f5b17be,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,64d1d3f0-1425-4094-9b28-3b06685add30,6752.644,2026-09-21T12:33:34.843Z,true,null,0.000000,null
f1d7ae12-5671-4ce0-af84-718a7365f2e9,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,cde55460-a3f1-4344-a3b9-45ac74bd6294,6579.206,2026-09-21T12:33:34.843Z,true,null,0.000000,null
7fc3ea83-7f8e-4cf2-b278-27bf7dcaeee2,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,2e4bd348-c5cd-4013-99a0-5cc72c9546cf,7312.570,2026-09-21T12:33:34.843Z,true,null,0.000000,null
6bfc6794-ea3a-4cad-a547-e0cc296af971,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,17fb4754-1b26-49de-a3b1-b594387bd1fe,3871.602,2026-09-21T12:33:34.843Z,true,null,0.000000,null
50141a1a-83de-45bf-a742-38ca6353bade,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,bf496185-f310-4511-898d-56507e20f34a,4497.601,2026-09-21T12:33:34.843Z,true,null,0.000000,null
ce8f6bc9-39a2-4cb2-95e1-dbb17e3eefbc,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,2f3438bd-72a1-4b98-891e-b71f383a0bcb,6640.907,2026-09-21T12:33:34.843Z,true,null,0.000000,null
482bfb89-aea6-4f11-99be-cd36e8f82e94,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,0ab897c6-ff3a-4d26-a5b9-876818222f79,8032.213,2026-09-21T12:33:34.843Z,true,null,0.000000,null
fb59fb3f-8909-4b53-b337-425fbc98bca6,dc2e31e4-147d-4737-9c48-f7a9cb298bdd,3992a714-8cde-43c2-bf96-ab327b0d838c,8343.467,2026-09-21T12:33:34.843Z,true,null,0.000000,null


###Produtividade

In [0]:
df_kpi_cooperado_mes = (
    coletas_coop
    .filter(F.col("cooperativa_id").isNotNull())
    .groupBy("cooperado_id", "cooperativa_id", "mes_referencia")
    .agg(F.count("*").alias("qtd_coletas"),
         F.sum("quantidade_kg").alias("produtividade_individual_kg"))
    .withColumn("participacao_cooperado_pct",
                F.col("produtividade_individual_kg")
                / F.sum("produtividade_individual_kg").over(
                    Window.partitionBy("cooperativa_id", "mes_referencia")) * 100)
)

In [0]:
display(df_kpi_cooperado_mes)

cooperado_id,cooperativa_id,mes_referencia,qtd_coletas,produtividade_individual_kg,participacao_cooperado_pct
4231d081-c8ea-54f5-becf-72bc5b111d36,0dd6255e-f26a-5842-88e5-6a9d915d214d,2026-08-01,1,280.000,100.00000000000000
7f9bdd91-7566-4fd3-b684-c13af73d3ab1,2415a263-6475-449d-858c-49135f5ab0ac,2026-05-01,1,192.176,76.94888786562294
d705669c-1fa3-4456-b131-8491a879c65d,2415a263-6475-449d-858c-49135f5ab0ac,2026-05-01,1,57.569,23.05111213437706
93881dcd-b87f-40fc-84c9-e7a112aca821,2415a263-6475-449d-858c-49135f5ab0ac,2026-06-01,1,111.456,14.61499526628150
deab7403-9d78-485c-9072-678394b507e7,2415a263-6475-449d-858c-49135f5ab0ac,2026-06-01,1,178.553,23.41328640701587
d705669c-1fa3-4456-b131-8491a879c65d,2415a263-6475-449d-858c-49135f5ab0ac,2026-06-01,2,373.138,48.92881588850978
6cb2bac2-5b61-414c-be68-218c71fb4262,2415a263-6475-449d-858c-49135f5ab0ac,2026-06-01,2,75.427,9.89058684996604
176e7f03-5258-4a4c-94f7-397c763ec211,2415a263-6475-449d-858c-49135f5ab0ac,2026-06-01,1,24.040,3.15231558822681
d705669c-1fa3-4456-b131-8491a879c65d,2415a263-6475-449d-858c-49135f5ab0ac,2026-07-01,1,88.676,9.28072206186171
6cb2bac2-5b61-414c-be68-218c71fb4262,2415a263-6475-449d-858c-49135f5ab0ac,2026-07-01,1,131.980,13.81286591326299


In [0]:

membros = (base["equipes_cooperados"]
           .groupBy("equipe_id")
           .agg(F.countDistinct("cooperado_id").alias("membros_equipe")))

df_kpi_equipe_mes = (
    triagens_coop
    .groupBy("equipe_id", "mes_referencia")
    .agg(F.sum("quantidade_kg").alias("kg_triado_equipe"),
         F.sum("rejeito_kg").alias("rejeito_kg_equipe"))
    .join(membros, "equipe_id", "left")                                       # 1:1 após agg
    .withColumn("taxa_aproveitamento_equipe_pct",
                F.when(F.col("kg_triado_equipe") > 0,
                       (F.col("kg_triado_equipe") - F.col("rejeito_kg_equipe"))
                       / F.col("kg_triado_equipe") * 100))
    .withColumn("kg_triado_por_membro", razao("kg_triado_equipe", "membros_equipe"))
)

In [0]:
display(df_kpi_equipe_mes)

equipe_id,mes_referencia,kg_triado_equipe,rejeito_kg_equipe,membros_equipe,taxa_aproveitamento_equipe_pct,kg_triado_por_membro
18cd4b90-803d-4341-bcac-c23c639e26f7,2026-09-01,1583.976,122.700,8,92.2536705102,197.997
215bf37e-00f4-4d6d-a293-23410cbef1a7,2026-09-01,694.346,51.000,9,92.6549587670,77.14955555555555
f8954173-2e88-4b0c-9edb-8dd90ce6e651,2026-09-01,1182.438,114.910,8,90.2819429010,147.80475
d6927be8-0c0a-4ddd-a560-b87ddf09d373,2026-09-01,1957.463,119.610,8,93.8895396746,244.682875
b586c18d-42f3-4315-a4da-866b96bb315f,2026-09-01,863.925,0.000,8,100.0000000000,107.990625
11991849-b7d3-4dc1-b8f9-6abe04a1c856,2026-09-01,2883.977,151.820,9,94.7357416512,320.4418888888889
0e415f33-3cc1-432a-87d8-8d27f8f1c088,2026-09-01,941.211,57.870,9,93.8515380717,104.579
5a67a8e3-7733-4146-a1e7-5e8a04676913,2026-09-01,56.624,0.000,8,100.0000000000,7.078
9aed27b0-e27c-4a3b-b61a-01c5c86b5748,2026-09-01,2046.318,117.130,8,94.2760607100,255.78975
8e8d355b-ef24-47d4-bccd-53c4234e7815,2026-09-01,1434.762,0.000,9,100.0000000000,159.418


##Tabelas finais

In [0]:
tabelas_finais = {
    "kpi_plataforma_mes": (df_kpi_plataforma_mes, ["mes_referencia"]),
    "kpi_cooperativa_mes": (df_kpi_cooperativa_mes, ["cooperativa_id", "mes_referencia"]),
    "kpi_cooperado_mes": (df_kpi_cooperado_mes, ["cooperado_id", "cooperativa_id", "mes_referencia"]),
    "kpi_equipe_mes": (df_kpi_equipe_mes, ["equipe_id", "mes_referencia"]),
    "kpi_estoque": (df_kpi_estoque, ["cooperativa_id", "material_id"]),
}

for nome, (df, grao) in tabelas_finais.items():
    dup = df.groupBy(*grao).count().filter(F.col("count") > 1).count()
    print(f"{nome}: {df.count()} linhas | chaves duplicadas no grão {grao}: {dup}")
    assert dup == 0, f"Granularidade violada em {nome}"
    # df.write.mode("overwrite").saveAsTable(f"<catalogo>.<schema>.{nome}")

kpi_plataforma_mes: 7 linhas | chaves duplicadas no grão ['mes_referencia']: 0
kpi_cooperativa_mes: 137 linhas | chaves duplicadas no grão ['cooperativa_id', 'mes_referencia']: 0
kpi_cooperado_mes: 379 linhas | chaves duplicadas no grão ['cooperado_id', 'cooperativa_id', 'mes_referencia']: 0
kpi_equipe_mes: 39 linhas | chaves duplicadas no grão ['equipe_id', 'mes_referencia']: 0
kpi_estoque: 1101 linhas | chaves duplicadas no grão ['cooperativa_id', 'material_id']: 0


## Publicação para BI (workspace.renovai_bi)
Dimensões (filtros/mapas), fatos (histogramas/boxplots) e KPIs mensais com comparativos (cards, barras e linhas), separados por empresa e cooperativa.

In [0]:
# Camada de consumo para os dashboards (Empresa e Cooperativa).
# Tabelas gravadas como Delta no Unity Catalog: workspace.renovai_bi.<tabela>
CATALOGO = "workspace"
SCHEMA_BI = "renovai_bi"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA_BI}")


def publicar(df, nome):
    destino = f"{CATALOGO}.{SCHEMA_BI}.{nome}"
    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(destino))
    print(f"✓ {destino}: {spark.table(destino).count()} linhas")


def add_comparativos(df, chave_entidade, colunas):
    """Colunas para cards com desvio:
       <kpi>_mes_anterior, <kpi>_var_pct (ou _var_pp para taxas em %),
       <kpi>_media_mes (média entre entidades no mês) e <kpi>_desvio_media_pct."""
    w_ent = Window.partitionBy(chave_entidade).orderBy("mes_referencia")
    w_mes = Window.partitionBy("mes_referencia")
    for c in colunas:
        v = F.col(c).cast("double")
        ant = F.lag(v).over(w_ent)
        media = F.avg(v).over(w_mes)
        df = df.withColumn(f"{c}_mes_anterior", ant)
        if c.endswith("_pct"):
            df = df.withColumn(f"{c}_var_pp", v - F.col(f"{c}_mes_anterior"))
        else:
            df = df.withColumn(
                f"{c}_var_pct",
                F.when(F.col(f"{c}_mes_anterior") > 0,
                       (v - F.col(f"{c}_mes_anterior")) / F.col(f"{c}_mes_anterior") * 100))
        df = df.withColumn(f"{c}_media_mes", media)
        df = df.withColumn(
            f"{c}_desvio_media_pct",
            F.when(F.col(f"{c}_media_mes") != 0,
                   (v - F.col(f"{c}_media_mes")) / F.abs(F.col(f"{c}_media_mes")) * 100))
    return df

In [0]:
end = base["enderecos"].select("endereco_id", "cidade", "bairro", "cep")
perf = base["perfis"].select(
    "empresa_id", "cooperativa_id", "endereco_id",
    F.col("data_criacao").alias("data_cadastro"),
    F.col("esta_ativo").alias("perfil_ativo"))

# perfis.empresa_id e perfis.cooperativa_id são UNIQUE -> joins 1:1
dim_empresa = (
    base["empresas"].select("empresa_id", F.col("nome").alias("empresa_nome"))
    .join(perf.filter(F.col("empresa_id").isNotNull()).drop("cooperativa_id"), "empresa_id", "left")
    .join(end, "endereco_id", "left")                                           # N:1
    .drop("endereco_id")
    .withColumnRenamed("cidade", "empresa_cidade")
    .withColumnRenamed("bairro", "empresa_bairro")
    .withColumnRenamed("cep", "empresa_cep")
)

dim_cooperativa = (
    base["cooperativas"].select("cooperativa_id", F.col("nome").alias("cooperativa_nome"),
                                "numero_cooperados", "contato_preferencial")
    .join(perf.filter(F.col("cooperativa_id").isNotNull()).drop("empresa_id"), "cooperativa_id", "left")
    .join(end, "endereco_id", "left")                                           # N:1
    .drop("endereco_id")
    .withColumnRenamed("cidade", "cooperativa_cidade")
    .withColumnRenamed("bairro", "cooperativa_bairro")
    .withColumnRenamed("cep", "cooperativa_cep")
)

cat = base["categorias_materiais"].select(
    "categoria_id", F.col("nome_categoria").alias("categoria"), "categoria_pai_id")
cat_pai = base["categorias_materiais"].select(
    F.col("categoria_id").alias("categoria_pai_id"), F.col("nome_categoria").alias("categoria_pai"))

dim_material = (
    base["materiais"].select("material_id", "categoria_id",
                             F.col("cooperativa_id").alias("material_cooperativa_id"),
                             "preco_sugerido", "esta_disponivel")
    .join(cat, "categoria_id", "left")                                          # N:1
    .join(cat_pai, "categoria_pai_id", "left")                                  # N:1
    .withColumn("categoria_grupo", F.coalesce("categoria_pai", "categoria"))
)

dim_calendario = (
    df_calendario
    .withColumn("ano", F.year("mes_referencia"))
    .withColumn("mes", F.month("mes_referencia"))
    .withColumn("ano_mes", F.date_format("mes_referencia", "yyyy-MM"))
)

emp_lk = dim_empresa.select("empresa_id", "empresa_nome", "empresa_cidade")
coop_lk = dim_cooperativa.select("cooperativa_id", "cooperativa_nome", "cooperativa_cidade")

In [0]:
# ---- Pedidos (grão: pedido) ----
itens_ped = (base["pedido_itens"].groupBy("pedido_id")
             .agg(F.count("*").alias("qtd_itens"),
                  F.sum(F.col("quantidade_kg") * F.col("preco_unitario")).alias("valor_estimado_pedido")))
direc = (base["pedidos_cooperativas"].groupBy("pedido_id")
         .agg(F.countDistinct("cooperativa_id").alias("qtd_cooperativas_direcionadas")))

fato_pedidos = (
    df_pedidos_base
    .join(itens_ped, "pedido_id", "left")                                       # 1:1 após agg
    .join(direc, "pedido_id", "left")                                           # 1:1 após agg
    .join(F.broadcast(emp_lk), "empresa_id", "left")                            # N:1
    .select("pedido_id", "empresa_id", "empresa_nome", "empresa_cidade",
            "data_pedido", "data_conclusao", F.col("mes_pedido").alias("mes_referencia"),
            "qtd_itens", "kg_pedido", "valor_estimado_pedido", "kg_negociado_sucesso",
            "qtd_negociacoes", "qtd_negociacoes_sucesso", "qtd_cooperativas_direcionadas",
            "pedido_concluido", "pedido_convertido", "tempo_conclusao_horas")
    .withColumn("taxa_atendimento_volume_pct", razao("kg_negociado_sucesso", "kg_pedido", pct=True))
)

# ---- Negociações (grão: negociação = pedido x cooperativa) ----
fato_negociacoes = (
    df_negociacoes_base
    .join(base["pedidos"].select("pedido_id", "data_pedido"), "pedido_id", "left")     # N:1
    .join(F.broadcast(emp_lk.select("empresa_id", "empresa_nome")), "empresa_id", "left")
    .join(F.broadcast(coop_lk.select("cooperativa_id", "cooperativa_nome")), "cooperativa_id", "left")
    .withColumn("tempo_ate_negociacao_horas", horas_entre("data_pedido", "data_inicio"))
    .select("negociacao_id", "pedido_id", "empresa_id", "empresa_nome",
            "cooperativa_id", "cooperativa_nome", "status_atual",
            "negociacao_sucesso", "negociacao_encerrada",
            "data_inicio", "data_fechamento", "mes_inicio", "mes_fechamento",
            "valor_negociacao", "kg_negociado", "qtd_contrapropostas",
            "tempo_negociacao_horas", "tempo_ate_negociacao_horas")
)

# ---- Itens negociados (grão: item da negociação) -> filtro por material ----
fato_itens_negociados = (
    base["negociacao_itens"]
    .select("negociacao_item_id", "negociacao_id", "material_id", "quantidade_kg", "preco_unitario")
    .join(fato_negociacoes.select("negociacao_id", "empresa_id", "empresa_nome", "cooperativa_id",
                                  "cooperativa_nome", "status_atual", "negociacao_sucesso",
                                  "mes_inicio", "mes_fechamento"),
          "negociacao_id", "inner")                                             # N:1
    .join(F.broadcast(dim_material.select("material_id", "categoria", "categoria_grupo", "preco_sugerido")),
          "material_id", "left")                                                # N:1
    .withColumn("valor_item", F.col("quantidade_kg") * F.col("preco_unitario"))
    .withColumn("desvio_preco_sugerido_pct",
                F.when(F.col("preco_sugerido") > 0,
                       (F.col("preco_unitario") - F.col("preco_sugerido")) / F.col("preco_sugerido") * 100))
)

# ---- Coletas (grão: coleta) ----
fato_coletas = (
    coletas_coop
    .join(F.broadcast(df_status.select("status_id", F.col("status_atual").alias("status_coleta"))),
          "status_id", "left")                                                  # N:1
    .join(F.broadcast(coop_lk.select("cooperativa_id", "cooperativa_nome")), "cooperativa_id", "left")
    .select(F.col("evento_id").alias("coleta_id"), "cooperativa_id", "cooperativa_nome",
            "cooperado_id", "rota_id", "tipo_coleta", "status_coleta",
            F.col("data_evento").alias("data_coleta"), "mes_referencia", "quantidade_kg")
)

# ---- Triagens (grão: triagem) ----
fato_triagens = (
    triagens_coop
    .join(F.broadcast(df_status.select("status_id", F.col("status_atual").alias("status_triagem"))),
          "status_id", "left")                                                  # N:1
    .join(F.broadcast(coop_lk.select("cooperativa_id", "cooperativa_nome")), "cooperativa_id", "left")
    .join(F.broadcast(dim_material.select("material_id", "categoria", "categoria_grupo")),
          "material_id", "left")                                                # N:1
    .withColumn("tempo_coleta_triagem_horas", horas_entre("data_coleta", "data_evento"))
    .withColumn("taxa_aproveitamento_pct",
                F.when(F.col("quantidade_kg") > 0,
                       (F.col("quantidade_kg") - F.col("rejeito_kg")) / F.col("quantidade_kg") * 100))
    .select(F.col("evento_id").alias("triagem_id"), "coleta_id", "cooperativa_id", "cooperativa_nome",
            "equipe_id", "material_id", "categoria", "categoria_grupo", "status_triagem",
            F.col("data_evento").alias("data_triagem"), "mes_referencia", "quantidade_kg",
            F.col("rejeito_kg").alias("quantidade_rejeito_kg"), "quantidade_aproveitada_kg",
            "taxa_aproveitamento_pct", "tempo_coleta_triagem_horas")
)

In [0]:
ped_emp = (
    df_pedidos_base
    .groupBy("empresa_id", F.col("mes_pedido").alias("mes_referencia"))
    .agg(F.count("*").alias("qtd_pedidos"),
         F.sum(F.col("pedido_concluido").cast("int")).alias("pedidos_concluidos"),
         F.sum(F.col("pedido_convertido").cast("int")).alias("pedidos_convertidos"),
         F.avg("tempo_conclusao_horas").alias("tempo_medio_conclusao_horas"),
         F.sum("kg_pedido").alias("kg_pedido"),
         F.sum("kg_negociado_sucesso").alias("kg_atendido"))
)

comp_emp = (
    df_negociacoes_base
    .filter(F.col("negociacao_encerrada"))
    .groupBy("empresa_id", F.col("mes_fechamento").alias("mes_referencia"))
    .agg(F.sum(F.when(F.col("negociacao_sucesso"), F.col("valor_negociacao"))).alias("valor_comprado"),
         F.sum(F.col("negociacao_sucesso").cast("int")).alias("compras_fechadas"),
         F.count("*").alias("negociacoes_encerradas"),
         F.countDistinct(F.when(F.col("negociacao_sucesso"), F.col("cooperativa_id")))
          .alias("cooperativas_fornecedoras"),
         F.avg("tempo_negociacao_horas").alias("tempo_medio_negociacao_horas"),
         F.avg("qtd_contrapropostas").alias("media_contrapropostas_negociacao"))
)

# Grade completa empresa x mês a partir do 1º mês de atividade da empresa
# (meses sem movimento entram com 0 -> lag compara meses adjacentes)
inicio_emp = (ped_emp.select("empresa_id", "mes_referencia")
              .unionByName(comp_emp.select("empresa_id", "mes_referencia"))
              .groupBy("empresa_id").agg(F.min("mes_referencia").alias("mes_inicio_atividade")))
grade_emp = (inicio_emp.crossJoin(F.broadcast(df_calendario))
             .filter(F.col("mes_referencia") >= F.col("mes_inicio_atividade"))
             .drop("mes_inicio_atividade"))

chave_emp = ["empresa_id", "mes_referencia"]
df_kpi_empresa_mes = (
    grade_emp
    .join(ped_emp, chave_emp, "left")                                           # 1:1
    .join(comp_emp, chave_emp, "left")                                          # 1:1
    .fillna(0, subset=["qtd_pedidos", "pedidos_concluidos", "pedidos_convertidos", "kg_pedido",
                       "kg_atendido", "valor_comprado", "compras_fechadas",
                       "negociacoes_encerradas", "cooperativas_fornecedoras"])
    .withColumn("taxa_conclusao_pedidos_pct", razao("pedidos_concluidos", "qtd_pedidos", pct=True))
    .withColumn("taxa_conversao_pedidos_pct", razao("pedidos_convertidos", "qtd_pedidos", pct=True))
    .withColumn("taxa_atendimento_volume_pct", razao("kg_atendido", "kg_pedido", pct=True))
    .withColumn("taxa_fechamento_negociacoes_pct",
                razao("compras_fechadas", "negociacoes_encerradas", pct=True))
    .withColumn("ticket_medio_compra", razao("valor_comprado", "compras_fechadas"))
    .withColumn("participacao_valor_comprado_pct",
                F.when(F.sum("valor_comprado").over(Window.partitionBy("mes_referencia")) > 0,
                       F.col("valor_comprado")
                       / F.sum("valor_comprado").over(Window.partitionBy("mes_referencia")) * 100))
    .join(F.broadcast(emp_lk), "empresa_id", "left")                            # N:1
)

df_kpi_empresa_mes = add_comparativos(
    df_kpi_empresa_mes, "empresa_id",
    ["qtd_pedidos", "valor_comprado", "ticket_medio_compra", "taxa_conversao_pedidos_pct",
     "taxa_atendimento_volume_pct", "tempo_medio_conclusao_horas", "kg_pedido"])

In [0]:
inicio_coop = (df_kpi_cooperativa_mes.groupBy("cooperativa_id")
               .agg(F.min("mes_referencia").alias("mes_inicio_atividade")))
grade_coop = (inicio_coop.crossJoin(F.broadcast(df_calendario))
              .filter(F.col("mes_referencia") >= F.col("mes_inicio_atividade"))
              .drop("mes_inicio_atividade"))

kpi_cooperativa_bi = (
    grade_coop
    .join(df_kpi_cooperativa_mes, ["cooperativa_id", "mes_referencia"], "left")  # 1:1
    .fillna(0, subset=contagens + ["valor_negociado"])
    .join(F.broadcast(coop_lk), "cooperativa_id", "left")                       # N:1
)

kpi_cooperativa_bi = add_comparativos(
    kpi_cooperativa_bi, "cooperativa_id",
    ["volume_coletado_kg", "volume_triado_kg", "taxa_rejeito_triagem_pct", "valor_negociado",
     "taxa_fechamento_negociacoes_pct", "kg_coletado_por_cooperado", "receita_por_cooperado",
     "tempo_medio_negociacao_horas"])

In [0]:
kpi_cooperado_bi = df_kpi_cooperado_mes.join(
    F.broadcast(coop_lk.select("cooperativa_id", "cooperativa_nome")), "cooperativa_id", "left")

# Cooperativa predominante de cada equipe (pela quantidade triada)
w_eq = Window.partitionBy("equipe_id").orderBy(F.col("kg").desc())
equipe_coop = (
    triagens_coop.filter(F.col("cooperativa_id").isNotNull())
    .groupBy("equipe_id", "cooperativa_id").agg(F.sum("quantidade_kg").alias("kg"))
    .withColumn("_rn", F.row_number().over(w_eq)).filter(F.col("_rn") == 1)
    .select("equipe_id", "cooperativa_id")
)
kpi_equipe_bi = (
    df_kpi_equipe_mes
    .join(base["equipes"].select("equipe_id", F.col("nome").alias("equipe_nome")), "equipe_id", "left")  # N:1
    .join(equipe_coop, "equipe_id", "left")                                     # N:1
    .join(F.broadcast(coop_lk.select("cooperativa_id", "cooperativa_nome")), "cooperativa_id", "left")
)

kpi_estoque_bi = (
    df_kpi_estoque
    .join(F.broadcast(coop_lk.select("cooperativa_id", "cooperativa_nome")), "cooperativa_id", "left")
    .join(F.broadcast(dim_material.select("material_id", "categoria", "categoria_grupo")),
          "material_id", "left")
)

## KPIs revisados 
Implementa a revisão dos KPIs com foco em decisão (Empresa, Cooperativa e Aplicativo):
funil por etapa, pedidos em aberto, taxa de resposta da cooperativa, material aguardando triagem,
estoque sem giro, rejeito por rota, ativação/risco de empresas, cadastros e acesso recente.
KPIs removidos do conjunto principal continuam como colunas (não quebram consultas), mas saem dos dashboards.

In [0]:
# ---------------- Parâmetros da revisão ----------------
STATUS_NEGOCIACAO_RECUSA = ["NEGOCIAÇÃO RECUSADA"]   # referencia NEGOCIACAO (fato_negociacoes.status_atual)      # status_atual de negociação recusada
DIAS_RISCO_EMPRESA = 60                     # empresa sem pedido há mais de N dias = em risco
TOP_N = 5                                   # concentração: top N participantes

prox_mes = F.add_months(F.col("mes_referencia"), 1).cast("timestamp")   # início do mês seguinte
ref_dia = F.least(F.date_sub(F.add_months(F.col("mes_referencia"), 1), 1), F.current_date())  # último dia considerado


def mediana(col):
    return F.percentile_approx(F.col(col), 0.5)


# Direcionamentos e negociações no grão (pedido, cooperativa) — chaves UNIQUE nas duas tabelas
direcionados = base["pedidos_cooperativas"].select("pedido_id", "cooperativa_id").distinct()
negociados = base["negociacoes"].select("pedido_id", "cooperativa_id").distinct()

ped_direc = direcionados.groupBy("pedido_id").agg(F.count("*").alias("qtd_direcionamentos"))
ped_neg = (df_negociacoes_base.groupBy("pedido_id")
           .agg(F.min("data_inicio").alias("data_primeira_negociacao")))
ped_acordo = (df_negociacoes_base.filter(F.col("negociacao_sucesso"))
              .groupBy("pedido_id")
              .agg(F.max("data_fechamento").alias("data_fechamento_acordo"),
                   F.sum("kg_negociado").alias("kg_acordado")))
ped_exped = (base["movimentacoes_estoques"]
             .filter((F.col("quantidade_kg") < 0) & F.col("item_id").isNotNull())
             .join(base["pedido_itens"].select("item_id", "pedido_id"), "item_id", "inner")   # N:1
             .groupBy("pedido_id").agg(F.sum(F.abs("quantidade_kg")).alias("kg_expedido")))
ped_aval = base["avaliacoes"].select("pedido_id").distinct().withColumn("foi_avaliado", F.lit(True))

df_pedidos_funil = (
    df_pedidos_base
    .select("pedido_id", "empresa_id", "data_pedido", "data_conclusao", "mes_pedido", "kg_pedido",
            "qtd_negociacoes", "pedido_convertido", "pedido_concluido", "tempo_conclusao_horas")
    .join(ped_direc, "pedido_id", "left")          # 1:1 após agg
    .join(ped_neg, "pedido_id", "left")            # 1:1 após agg
    .join(ped_acordo, "pedido_id", "left")         # 1:1 após agg
    .join(ped_exped, "pedido_id", "left")          # 1:1 após agg
    .join(ped_aval, "pedido_id", "left")           # 1:1
    .withColumn("foi_direcionado", F.coalesce("qtd_direcionamentos", F.lit(0)) > 0)
    .withColumn("foi_negociado", F.coalesce("qtd_negociacoes", F.lit(0)) > 0)
    .withColumn("teve_acordo", F.coalesce("pedido_convertido", F.lit(False)))
    # entrega só conta para pedidos com acordo; saídas ligadas a pedidos sem acordo são sinalizadas à parte
    .withColumn("kg_expedido_acordo", F.when(F.col("teve_acordo"), F.col("kg_expedido")))
    .withColumn("foi_entregue", F.col("teve_acordo") & (F.coalesce("kg_expedido", F.lit(0)) > 0))
    .withColumn("expedido_sem_acordo", ~F.col("teve_acordo") & (F.coalesce("kg_expedido", F.lit(0)) > 0))
    .withColumn("foi_avaliado", F.coalesce("foi_avaliado", F.lit(False)))
    .withColumn("conclusao_com_acordo", F.col("pedido_concluido") & F.col("teve_acordo"))
    .withColumn("encerrado_sem_acordo", F.col("pedido_concluido") & ~F.col("teve_acordo"))
    .withColumn("tempo_ate_negociacao_horas", horas_entre("data_pedido", "data_primeira_negociacao"))
    .withColumn("tempo_fechamento_conclusao_horas",
                F.when(F.col("teve_acordo") & F.col("data_conclusao").isNotNull(),
                       horas_entre("data_fechamento_acordo", "data_conclusao")))
    .withColumn("tempo_conclusao_com_acordo_horas",
                F.when(F.col("conclusao_com_acordo"), F.col("tempo_conclusao_horas")))
)

# Intervalo entre pedidos consecutivos da mesma empresa
w_ped_emp = Window.partitionBy("empresa_id").orderBy("data_pedido")
df_pedidos_funil = df_pedidos_funil.withColumn(
    "dias_desde_pedido_anterior",
    F.datediff(F.col("data_pedido"), F.lag("data_pedido").over(w_ped_emp)))

In [0]:
neg_tempo_mes = (df_negociacoes_base
                 .join(base["pedidos"].select("pedido_id", "data_pedido"), "pedido_id", "left")   # N:1
                 .groupBy(mes("data_pedido").alias("mes_referencia"))
                 .agg(mediana("tempo_negociacao_horas").alias("mediana_tempo_negociacao_horas")))

df_funil_mes = (
    df_pedidos_funil
    .groupBy(F.col("mes_pedido").alias("mes_referencia"))
    .agg(F.count("*").alias("qtd_pedidos"),
         F.sum(F.col("foi_direcionado").cast("int")).alias("pedidos_direcionados"),
         F.sum(F.col("foi_negociado").cast("int")).alias("pedidos_negociados"),
         F.sum(F.col("teve_acordo").cast("int")).alias("pedidos_com_acordo"),
         F.sum(F.col("foi_entregue").cast("int")).alias("pedidos_entregues"),
         F.sum(F.col("conclusao_com_acordo").cast("int")).alias("pedidos_concluidos_com_acordo"),
         F.sum(F.col("encerrado_sem_acordo").cast("int")).alias("pedidos_encerrados_sem_acordo"),
         F.sum("kg_pedido").alias("kg_pedido"),
         F.sum("kg_acordado").alias("kg_acordado"),
         F.sum("kg_expedido_acordo").alias("kg_expedido"),
         F.sum("kg_expedido").alias("kg_expedido_total"),
         F.sum(F.col("expedido_sem_acordo").cast("int")).alias("pedidos_expedidos_sem_acordo"),
         mediana("tempo_ate_negociacao_horas").alias("mediana_tempo_ate_negociacao_horas"),
         mediana("tempo_fechamento_conclusao_horas").alias("mediana_tempo_fechamento_conclusao_horas"),
         mediana("tempo_conclusao_com_acordo_horas").alias("mediana_tempo_conclusao_com_acordo_horas"))
    .join(neg_tempo_mes, "mes_referencia", "left")                                           # 1:1
    .join(F.broadcast(df_calendario), "mes_referencia", "right")                              # grade completa
    .fillna(0, subset=["qtd_pedidos", "pedidos_direcionados", "pedidos_negociados", "pedidos_com_acordo",
                       "pedidos_entregues", "pedidos_concluidos_com_acordo", "pedidos_encerrados_sem_acordo",
                       "pedidos_expedidos_sem_acordo"])
    # taxa de cada etapa em relação à etapa anterior
    .withColumn("pct_etapa_direcionado", razao("pedidos_direcionados", "qtd_pedidos", pct=True))
    .withColumn("pct_etapa_negociacao", razao("pedidos_negociados", "pedidos_direcionados", pct=True))
    .withColumn("pct_etapa_acordo", razao("pedidos_com_acordo", "pedidos_negociados", pct=True))
    .withColumn("pct_etapa_entregue", razao("pedidos_entregues", "pedidos_com_acordo", pct=True))
    # taxas sobre o total de pedidos (para o gráfico de funil)
    .withColumn("pct_pedidos_negociados", razao("pedidos_negociados", "qtd_pedidos", pct=True))
    .withColumn("pct_pedidos_com_acordo", razao("pedidos_com_acordo", "qtd_pedidos", pct=True))
    .withColumn("taxa_entrega_fisica_pct", razao("kg_expedido", "kg_acordado", pct=True))
)

In [0]:
emp_ped_rev = (
    df_pedidos_funil
    .groupBy("empresa_id", F.col("mes_pedido").alias("mes_referencia"))
    .agg(F.sum((~F.col("foi_negociado")).cast("int")).alias("pedidos_sem_negociacao"),
         F.sum(F.col("conclusao_com_acordo").cast("int")).alias("pedidos_concluidos_com_acordo"),
         F.sum(F.col("encerrado_sem_acordo").cast("int")).alias("pedidos_encerrados_sem_acordo"),
         mediana("tempo_conclusao_com_acordo_horas").alias("mediana_tempo_conclusao_horas"),
         F.avg("dias_desde_pedido_anterior").alias("intervalo_medio_entre_pedidos_dias"))
)

emp_neg_rev = (
    df_negociacoes_base.filter(F.col("negociacao_encerrada"))
    .withColumn("negociacao_recusada", flag_status("status_atual", STATUS_NEGOCIACAO_RECUSA))
    .groupBy("empresa_id", F.col("mes_fechamento").alias("mes_referencia"))
    .agg(F.sum(F.col("negociacao_recusada").cast("int")).alias("negociacoes_recusadas"))
)

forn = (df_negociacoes_base.filter(F.col("negociacao_sucesso"))
        .groupBy("empresa_id", F.col("mes_fechamento").alias("mes_referencia"), "cooperativa_id")
        .agg(F.sum("valor_negociacao").alias("v")))
emp_forn = (forn.groupBy("empresa_id", "mes_referencia")
            .agg(F.max("v").alias("valor_principal_fornecedor"), F.sum("v").alias("v_tot"))
            .withColumn("participacao_principal_fornecedor_pct", razao("valor_principal_fornecedor", "v_tot", pct=True))
            .drop("v_tot"))

emp_preco = (fato_itens_negociados.filter(F.col("negociacao_sucesso") & F.col("desvio_preco_sugerido_pct").isNotNull())
             .groupBy("empresa_id", F.col("mes_fechamento").alias("mes_referencia"))
             .agg((F.sum(F.col("desvio_preco_sugerido_pct") * F.col("quantidade_kg")) / F.sum("quantidade_kg"))
                  .alias("desvio_preco_medio_pct")))

# Pedidos em aberto e dias desde o último pedido, como foto no fim de cada mês
emp_aberto = (
    df_kpi_empresa_mes.select("empresa_id", "mes_referencia")
    .join(df_pedidos_funil.select("empresa_id", "data_pedido", "data_conclusao"), "empresa_id", "inner")
    .filter(F.col("data_pedido") < prox_mes)
    .withColumn("aberto", F.col("data_conclusao").isNull() | (F.col("data_conclusao") >= prox_mes))
    .groupBy("empresa_id", "mes_referencia")
    .agg(F.sum(F.col("aberto").cast("int")).alias("pedidos_em_aberto"),
         F.avg(F.when(F.col("aberto"), F.datediff(ref_dia, F.col("data_pedido")))).alias("idade_media_pedidos_abertos_dias"),
         F.max("data_pedido").alias("data_ultimo_pedido"))
    .withColumn("dias_desde_ultimo_pedido", F.datediff(ref_dia, F.col("data_ultimo_pedido")))
    .withColumn("empresa_em_risco", F.col("dias_desde_ultimo_pedido") > DIAS_RISCO_EMPRESA)
)

chave_emp = ["empresa_id", "mes_referencia"]
df_kpi_empresa_mes = (
    df_kpi_empresa_mes
    .join(emp_ped_rev, chave_emp, "left")
    .join(emp_neg_rev, chave_emp, "left")
    .join(emp_forn, chave_emp, "left")
    .join(emp_preco, chave_emp, "left")
    .join(emp_aberto, chave_emp, "left")
    .fillna(0, subset=["pedidos_sem_negociacao", "pedidos_concluidos_com_acordo",
                       "pedidos_encerrados_sem_acordo", "negociacoes_recusadas", "pedidos_em_aberto"])
    .withColumn("taxa_pedidos_sem_negociacao_pct", razao("pedidos_sem_negociacao", "qtd_pedidos", pct=True))
    .withColumn("taxa_conclusao_com_acordo_pct", razao("pedidos_concluidos_com_acordo", "qtd_pedidos", pct=True))
    .withColumn("taxa_encerrado_sem_acordo_pct", razao("pedidos_encerrados_sem_acordo", "qtd_pedidos", pct=True))
    .withColumn("taxa_recusa_negociacoes_pct", razao("negociacoes_recusadas", "negociacoes_encerradas", pct=True))
)

# Atendimento do volume por categoria (empresa x categoria x mês do pedido)
cat_mat = dim_material.select("material_id", F.coalesce("categoria_grupo", F.lit("Não informado")).alias("categoria_grupo"))
ped_mes_emp = base["pedidos"].select("pedido_id", "empresa_id", mes("data_pedido").alias("mes_referencia"))
kg_ped_cat = (base["pedido_itens"].join(F.broadcast(cat_mat), "material_id", "left")
              .join(ped_mes_emp, "pedido_id", "inner")
              .withColumn("categoria_grupo", F.coalesce("categoria_grupo", F.lit("Não informado")))
              .groupBy("empresa_id", "categoria_grupo", "mes_referencia")
              .agg(F.sum("quantidade_kg").alias("kg_pedido")))
kg_at_cat = (base["negociacao_itens"]
             .join(df_negociacoes_base.filter(F.col("negociacao_sucesso")).select("negociacao_id", "pedido_id"),
                   "negociacao_id", "inner")
             .join(F.broadcast(cat_mat), "material_id", "left")
             .join(ped_mes_emp, "pedido_id", "inner")
             .withColumn("categoria_grupo", F.coalesce("categoria_grupo", F.lit("Não informado")))
              .groupBy("empresa_id", "categoria_grupo", "mes_referencia")
             .agg(F.sum("quantidade_kg").alias("kg_atendido")))
kpi_empresa_categoria_bi = (
    kg_ped_cat.join(kg_at_cat, ["empresa_id", "categoria_grupo", "mes_referencia"], "full")
    .fillna(0, subset=["kg_pedido", "kg_atendido"])
    .withColumn("categoria_grupo", F.coalesce("categoria_grupo", F.lit("Não informado")))
    .withColumn("taxa_atendimento_categoria_pct", razao("kg_atendido", "kg_pedido", pct=True))
    .join(F.broadcast(emp_lk), "empresa_id", "left")
)

In [0]:
chave_coop = ["cooperativa_id", "mes_referencia"]

# Taxa de resposta: pedidos direcionados à cooperativa que viraram negociação com ela
coop_resp = (
    direcionados
    .join(negociados.withColumn("respondeu", F.lit(1)), ["pedido_id", "cooperativa_id"], "left")
    .join(base["pedidos"].select("pedido_id", mes("data_pedido").alias("mes_referencia")), "pedido_id", "inner")
    .groupBy(*chave_coop)
    .agg(F.count("*").alias("pedidos_recebidos"),
         F.sum(F.coalesce("respondeu", F.lit(0))).alias("pedidos_respondidos"))
)

# Material aguardando triagem (foto no fim de cada mês)
prim_triagem = (triagens_coop.groupBy("coleta_id").agg(F.min("data_evento").alias("data_primeira_triagem")))
coop_backlog = (
    coletas_coop.filter(F.col("cooperativa_id").isNotNull())
    .select(F.col("evento_id").alias("coleta_id"), "cooperativa_id", "data_evento", "quantidade_kg")
    .join(prim_triagem, "coleta_id", "left")
    .crossJoin(F.broadcast(df_calendario))
    .filter(F.col("data_evento") < prox_mes)
    .filter(F.col("data_primeira_triagem").isNull() | (F.col("data_primeira_triagem") >= prox_mes))
    .groupBy(*chave_coop)
    .agg(F.sum("quantidade_kg").alias("kg_aguardando_triagem"),
         F.avg(F.datediff(ref_dia, F.col("data_evento"))).alias("idade_media_backlog_triagem_dias"))
)

# Cooperados que coletaram no mês
coop_coletores = (coletas_coop.filter(F.col("cooperativa_id").isNotNull())
                  .groupBy(*chave_coop).agg(F.countDistinct("cooperado_id").alias("cooperados_coletores")))

# kg acordado pela cooperativa (mês de fechamento) para a taxa de entrega física
coop_acordo = (df_negociacoes_base.filter(F.col("negociacao_sucesso"))
               .groupBy("cooperativa_id", F.col("mes_fechamento").alias("mes_referencia"))
               .agg(F.sum("kg_negociado").alias("kg_acordado")))

# Avaliações recebidas. O tables.sql não declara a FK de avaliacoes.avaliado_id:
# testa se o id corresponde a perfis.perfil_id ou a usuarios.usuario_id e usa o que casar.
aval = base["avaliacoes"].filter(F.col("nota").between(1, 5))
m_perf = aval.join(base["perfis"].select(F.col("perfil_id").alias("avaliado_id")), "avaliado_id", "inner").count()
m_usu = aval.join(base["usuarios"].select(F.col("usuario_id").alias("avaliado_id")), "avaliado_id", "inner").count()
print(f"avaliacoes: {aval.count()} | avaliado_id casa com perfis: {m_perf} | com usuarios: {m_usu}")
if m_perf >= m_usu and m_perf > 0:
    mapa_avaliado = (base["perfis"].filter(F.col("cooperativa_id").isNotNull())
                     .select(F.col("perfil_id").alias("avaliado_id"), "cooperativa_id"))
elif m_usu > 0:
    mapa_avaliado = (base["funcionarios"].filter(F.col("cooperativa_id").isNotNull())
                     .select(F.col("usuario_id").alias("avaliado_id"), "cooperativa_id").dropDuplicates(["avaliado_id"]))
else:
    mapa_avaliado = None
if mapa_avaliado is not None:
    coop_aval = (aval.join(mapa_avaliado, "avaliado_id", "inner")
                 .groupBy("cooperativa_id", mes("data_avaliacao").alias("mes_referencia"))
                 .agg(F.avg("nota").alias("nota_media_recebida"), F.count("*").alias("avaliacoes_recebidas")))
else:
    coop_aval = spark.createDataFrame([], "cooperativa_id string, mes_referencia date, nota_media_recebida double, avaliacoes_recebidas long")

# kg expedido em pedidos com acordo fechado com a própria cooperativa
acordos_pc = df_negociacoes_base.filter(F.col("negociacao_sucesso")).select("pedido_id", "cooperativa_id").distinct()
coop_exp_ac = (base["movimentacoes_estoques"]
               .filter((F.col("quantidade_kg") < 0) & F.col("item_id").isNotNull())
               .join(base["estoques"].select("estoque_id", "cooperativa_id"), "estoque_id", "inner")     # N:1
               .join(base["pedido_itens"].select("item_id", "pedido_id"), "item_id", "inner")           # N:1
               .join(acordos_pc, ["pedido_id", "cooperativa_id"], "inner")
               .groupBy("cooperativa_id", mes("data_movimentacao").alias("mes_referencia"))
               .agg(F.sum(F.abs("quantidade_kg")).alias("kg_expedido_acordo")))

w_coop_hist = Window.partitionBy("cooperativa_id")
w_coop_3m = Window.partitionBy("cooperativa_id").orderBy("mes_referencia").rowsBetween(-2, 0)

kpi_cooperativa_bi = (
    kpi_cooperativa_bi
    .join(coop_resp, chave_coop, "left")
    .join(coop_backlog, chave_coop, "left")
    .join(coop_coletores, chave_coop, "left")
    .join(coop_acordo, chave_coop, "left")
    .join(coop_aval, chave_coop, "left")
    .join(coop_exp_ac, chave_coop, "left")
    .fillna(0, subset=["pedidos_recebidos", "pedidos_respondidos", "kg_aguardando_triagem",
                       "cooperados_coletores", "kg_acordado", "avaliacoes_recebidas", "kg_expedido_acordo"])
    .withColumn("taxa_resposta_pedidos_pct", razao("pedidos_respondidos", "pedidos_recebidos", pct=True))
    .withColumn("taxa_recusa_direcionamento_pct", F.col("taxa_cancelamento_pct"))       # renomeado
    .withColumn("taxa_cooperados_coletores_pct", razao("cooperados_coletores", "cooperados_ativos", pct=True))
    .withColumn("kg_por_cooperado_coletor", razao("volume_coletado_kg", "cooperados_coletores"))
    .withColumn("media_coleta_mensal_kg",
                F.avg(F.when(F.col("volume_coletado_kg") > 0, F.col("volume_coletado_kg"))).over(w_coop_hist))
    .withColumn("indice_carga_coleta", razao("volume_coletado_kg", "media_coleta_mensal_kg"))
    .withColumn("taxa_entrega_fisica_pct", razao("kg_expedido_acordo", "kg_acordado", pct=True))
    .withColumn("despesa_3m", F.sum("despesa_total").over(w_coop_3m))
    .withColumn("aproveitado_3m", F.sum("volume_aproveitado_kg").over(w_coop_3m))
    .withColumn("custo_por_kg_aproveitado_3m", razao("despesa_3m", "aproveitado_3m"))
)

# Rejeito por rota e tipo de coleta (grão: cooperativa x rota x tipo x mês da coleta)
col_rota = (coletas_coop.filter(F.col("cooperativa_id").isNotNull())
            .select(F.col("evento_id").alias("coleta_id"), "cooperativa_id",
                    F.coalesce(F.col("rota_id").cast("string"), F.lit("SEM_ROTA")).alias("rota_chave"),
                    F.coalesce("tipo_coleta", F.lit("Não informado")).alias("tipo_coleta"),
                    "mes_referencia", "quantidade_kg"))
rota_coleta = (col_rota.groupBy("cooperativa_id", "rota_chave", "tipo_coleta", "mes_referencia")
               .agg(F.count("*").alias("qtd_coletas"), F.sum("quantidade_kg").alias("kg_coletado")))
rota_triagem = (triagens_coop.select("coleta_id", F.col("quantidade_kg").alias("kg_tri"), "rejeito_kg")
                .join(col_rota.drop("quantidade_kg"), "coleta_id", "inner")
                .groupBy("cooperativa_id", "rota_chave", "tipo_coleta", "mes_referencia")
                .agg(F.sum("kg_tri").alias("kg_triado"), F.sum("rejeito_kg").alias("kg_rejeito")))
rotas_nome = base["rotas"].select(F.col("rota_id").cast("string").alias("rota_chave"), F.col("nome").alias("rota_nome"))
kpi_rota_bi = (
    rota_coleta.join(rota_triagem, ["cooperativa_id", "rota_chave", "tipo_coleta", "mes_referencia"], "left")
    .fillna(0, subset=["kg_triado", "kg_rejeito"])
    .join(F.broadcast(rotas_nome), "rota_chave", "left")
    .withColumn("rota_nome", F.coalesce("rota_nome", F.lit("Sem rota")))
    .withColumn("kg_medio_por_coleta", razao("kg_coletado", "qtd_coletas"))
    .withColumn("taxa_rejeito_rota_pct", razao("kg_rejeito", "kg_triado", pct=True))
    .join(F.broadcast(coop_lk), "cooperativa_id", "left")
)

# Equipe: uma só métrica de qualidade (rejeito)
kpi_equipe_bi = kpi_equipe_bi.withColumn("taxa_rejeito_equipe_pct",
                                         razao("rejeito_kg_equipe", "kg_triado_equipe", pct=True))

# Estoque sem giro em 90 dias (foto atual)
kpi_estoque_bi = (kpi_estoque_bi
                  .withColumn("sem_giro_90d", F.coalesce("saida_kg_janela", F.lit(0)) == 0)
                  .withColumn("kg_sem_giro_90d", F.when(F.col("sem_giro_90d"), F.col("quantidade_kg")).otherwise(F.lit(0))))

avaliacoes: 288 | avaliado_id casa com perfis: 288 | com usuarios: 0


In [0]:
perf_all = base["perfis"].select("empresa_id", "cooperativa_id", "data_criacao")
cad_mes = (perf_all.groupBy(mes("data_criacao").alias("mes_referencia"))
           .agg(F.sum(F.col("empresa_id").isNotNull().cast("int")).alias("novas_empresas"),
                F.sum(F.col("cooperativa_id").isNotNull().cast("int")).alias("novas_cooperativas")))
usu_mes = (base["usuarios"].groupBy(mes("data_criacao").alias("mes_referencia"))
           .agg(F.count("*").alias("novos_usuarios")))

prim_ped = df_pedidos_base.groupBy("empresa_id").agg(F.min("data_pedido").alias("data_primeiro_pedido"))
ativ = (perf_all.filter(F.col("empresa_id").isNotNull())
        .join(prim_ped, "empresa_id", "left")
        .groupBy(mes("data_criacao").alias("mes_referencia"))
        .agg(F.count("*").alias("empresas_cadastradas_coorte"),
             F.sum(F.col("data_primeiro_pedido").isNotNull().cast("int")).alias("empresas_ativadas_coorte"),
             F.avg(F.datediff("data_primeiro_pedido", "data_criacao")).alias("dias_ate_primeiro_pedido")))

risco = (df_kpi_empresa_mes.groupBy("mes_referencia")
         .agg(F.sum(F.col("empresa_em_risco").cast("int")).alias("empresas_em_risco")))

oper = (coletas_coop.filter(F.col("cooperativa_id").isNotNull()).select("cooperativa_id", "mes_referencia")
        .unionByName(triagens_coop.filter(F.col("cooperativa_id").isNotNull()).select("cooperativa_id", "mes_referencia"))
        .groupBy("mes_referencia").agg(F.countDistinct("cooperativa_id").alias("cooperativas_operando")))

plat_ped = (df_pedidos_funil.groupBy(F.col("mes_pedido").alias("mes_referencia"))
            .agg(F.sum(F.col("foi_direcionado").cast("int")).alias("pedidos_direcionados"),
                 F.avg(F.when(F.col("foi_direcionado"), F.col("qtd_direcionamentos"))).alias("media_cooperativas_por_pedido"),
                 F.sum(F.col("teve_acordo").cast("int")).alias("pedidos_com_acordo"),
                 F.sum((F.col("teve_acordo") & F.col("foi_avaliado")).cast("int")).alias("pedidos_com_acordo_avaliados")))


def top_n_pct(df_val, chave, col_valor, nome):
    w = Window.partitionBy("mes_referencia").orderBy(F.col(col_valor).desc())
    return (df_val.filter(F.col(col_valor) > 0)
            .withColumn("_rk", F.row_number().over(w))
            .groupBy("mes_referencia")
            .agg((F.sum(F.when(F.col("_rk") <= TOP_N, F.col(col_valor))) / F.sum(col_valor) * 100).alias(nome)))


top_emp = top_n_pct(df_kpi_empresa_mes, "empresa_id", "valor_comprado", "participacao_top5_empresas_pct")
top_coop = top_n_pct(kpi_cooperativa_bi, "cooperativa_id", "valor_negociado", "participacao_top5_cooperativas_pct")

w_plat = Window.orderBy("mes_referencia")
df_kpi_plataforma_mes = (
    df_kpi_plataforma_mes
    .join(cad_mes, "mes_referencia", "left")
    .join(usu_mes, "mes_referencia", "left")
    .join(ativ, "mes_referencia", "left")
    .join(risco, "mes_referencia", "left")
    .join(oper, "mes_referencia", "left")
    .join(plat_ped, "mes_referencia", "left")
    .join(top_emp, "mes_referencia", "left")
    .join(top_coop, "mes_referencia", "left")
    .fillna(0, subset=["novas_empresas", "novas_cooperativas", "novos_usuarios", "empresas_em_risco",
                       "cooperativas_operando", "pedidos_direcionados", "pedidos_com_acordo",
                       "pedidos_com_acordo_avaliados"])
    .withColumn("cooperativas_negociando", F.col("cooperativas_ativas"))                    # renomeado
    .withColumn("taxa_ativacao_empresas_pct", razao("empresas_ativadas_coorte", "empresas_cadastradas_coorte", pct=True))
    .withColumn("taxa_pedidos_direcionados_pct", razao("pedidos_direcionados", "qtd_pedidos", pct=True))
    .withColumn("taxa_avaliacao_pct", razao("pedidos_com_acordo_avaliados", "pedidos_com_acordo", pct=True))
    .withColumn("pedidos_media_3m_anteriores", F.avg("qtd_pedidos").over(w_plat.rowsBetween(-3, -1)))
    .withColumn("var_pedidos_media3m_pct", variacao_pct("qtd_pedidos", "pedidos_media_3m_anteriores"))
)

# Demanda x oferta por categoria (foto atual)
kg_sem_acordo = (base["pedido_itens"]
                 .join(df_pedidos_funil.filter(~F.col("teve_acordo")).select("pedido_id"), "pedido_id", "inner")
                 .join(F.broadcast(cat_mat), "material_id", "left")
                 .withColumn("categoria_grupo", F.coalesce("categoria_grupo", F.lit("Não informado"))).groupBy("categoria_grupo").agg(F.sum("quantidade_kg").alias("kg_pedido_sem_acordo")))
kg_est_cat = (base["estoques"].join(F.broadcast(cat_mat), "material_id", "left")
              .withColumn("categoria_grupo", F.coalesce("categoria_grupo", F.lit("Não informado"))).groupBy("categoria_grupo").agg(F.sum("quantidade_kg").alias("kg_estoque"),
                                              F.countDistinct("cooperativa_id").alias("cooperativas_com_estoque")))
kpi_categoria_bi = (kg_sem_acordo.join(kg_est_cat, "categoria_grupo", "full")
                    .withColumn("categoria_grupo", F.coalesce("categoria_grupo", F.lit("Não informado")))
                    .fillna(0, subset=["kg_pedido_sem_acordo", "kg_estoque"])
                    .withColumn("razao_demanda_oferta", razao("kg_pedido_sem_acordo", "kg_estoque"))
                    .withColumn("data_referencia", F.current_date()))

# Acesso recente de usuários: snapshot gravado em append (1 linha por dia de execução)
kpi_plataforma_snapshot = (
    base["usuarios"].filter(F.coalesce(F.col("esta_ativo"), F.lit(True)))
    .agg(F.count("*").alias("usuarios_ativos"),
         F.sum((F.col("ultimo_acesso") >= F.date_sub(F.current_date(), 30)).cast("int")).alias("usuarios_acesso_30d"))
    .withColumn("pct_usuarios_acesso_30d", razao("usuarios_acesso_30d", "usuarios_ativos", pct=True))
    .withColumn("data_referencia", F.current_date())
)


# Fato de pedidos com as etapas do funil (permite filtrar o funil por empresa)
fato_pedidos = fato_pedidos.join(
    df_pedidos_funil.select("pedido_id", "foi_direcionado", "foi_negociado", "teve_acordo", "foi_entregue",
                            "foi_avaliado", "conclusao_com_acordo", "encerrado_sem_acordo", "kg_acordado",
                            "kg_expedido", "kg_expedido_acordo", "expedido_sem_acordo", "tempo_ate_negociacao_horas", "tempo_fechamento_conclusao_horas",
                            "dias_desde_pedido_anterior"),
    "pedido_id", "left")                                                                     # 1:1

In [0]:
publicacao = {
    # dimensões (filtros e mapas)
    "dim_empresa": (dim_empresa, ["empresa_id"]),
    "dim_cooperativa": (dim_cooperativa, ["cooperativa_id"]),
    "dim_material": (dim_material, ["material_id"]),
    "dim_calendario": (dim_calendario, ["mes_referencia"]),
    # fatos (histogramas e boxplots)
    "fato_pedidos": (fato_pedidos, ["pedido_id"]),
    "fato_negociacoes": (fato_negociacoes, ["negociacao_id"]),
    "fato_itens_negociados": (fato_itens_negociados, ["negociacao_item_id"]),
    "fato_coletas": (fato_coletas, ["coleta_id"]),
    "fato_triagens": (fato_triagens, ["triagem_id"]),
    # KPIs (cards, barras e linhas)
    "kpi_plataforma_mes": (df_kpi_plataforma_mes, ["mes_referencia"]),
    "kpi_empresa_mes": (df_kpi_empresa_mes, ["empresa_id", "mes_referencia"]),
    "kpi_cooperativa_mes": (kpi_cooperativa_bi, ["cooperativa_id", "mes_referencia"]),
    "kpi_cooperado_mes": (kpi_cooperado_bi, ["cooperado_id", "cooperativa_id", "mes_referencia"]),
    "kpi_equipe_mes": (kpi_equipe_bi, ["equipe_id", "mes_referencia"]),
    "kpi_estoque": (kpi_estoque_bi, ["cooperativa_id", "material_id"]),
    # revisão de KPIs (02/10/2026)
    "kpi_funil_mes": (df_funil_mes, ["mes_referencia"]),
    "kpi_empresa_categoria_mes": (kpi_empresa_categoria_bi, ["empresa_id", "categoria_grupo", "mes_referencia"]),
    "kpi_rota_mes": (kpi_rota_bi, ["cooperativa_id", "rota_chave", "tipo_coleta", "mes_referencia"]),
    "kpi_categoria": (kpi_categoria_bi, ["categoria_grupo"]),
}

for nome, (df, grao) in publicacao.items():
    dup = df.groupBy(*grao).count().filter(F.col("count") > 1).count()
    assert dup == 0, f"Granularidade violada em {nome}: {dup} chaves duplicadas em {grao}"
    publicar(df, nome)

✓ workspace.renovai_bi.dim_empresa: 49 linhas
✓ workspace.renovai_bi.dim_cooperativa: 32 linhas
✓ workspace.renovai_bi.dim_material: 88 linhas
✓ workspace.renovai_bi.dim_calendario: 7 linhas
✓ workspace.renovai_bi.fato_pedidos: 268 linhas
✓ workspace.renovai_bi.fato_negociacoes: 84 linhas
✓ workspace.renovai_bi.fato_itens_negociados: 183 linhas
✓ workspace.renovai_bi.fato_coletas: 455 linhas
✓ workspace.renovai_bi.fato_triagens: 394 linhas
✓ workspace.renovai_bi.kpi_plataforma_mes: 7 linhas
✓ workspace.renovai_bi.kpi_empresa_mes: 278 linhas
✓ workspace.renovai_bi.kpi_cooperativa_mes: 135 linhas
✓ workspace.renovai_bi.kpi_cooperado_mes: 379 linhas
✓ workspace.renovai_bi.kpi_equipe_mes: 39 linhas
✓ workspace.renovai_bi.kpi_estoque: 1101 linhas
✓ workspace.renovai_bi.kpi_funil_mes: 7 linhas
✓ workspace.renovai_bi.kpi_empresa_categoria_mes: 432 linhas
✓ workspace.renovai_bi.kpi_rota_mes: 217 linhas
✓ workspace.renovai_bi.kpi_categoria: 12 linhas


In [0]:
destino_snap = f"{CATALOGO}.{SCHEMA_BI}.kpi_plataforma_snapshot"
if spark.catalog.tableExists(destino_snap):
    spark.sql(f"DELETE FROM {destino_snap} WHERE data_referencia = current_date()")
    kpi_plataforma_snapshot.write.mode("append").saveAsTable(destino_snap)
else:
    kpi_plataforma_snapshot.write.mode("overwrite").saveAsTable(destino_snap)
print(f"✓ {destino_snap}: {spark.table(destino_snap).count()} linhas")
display(spark.table(destino_snap).orderBy("data_referencia"))

✓ workspace.renovai_bi.kpi_plataforma_snapshot: 3 linhas


usuarios_ativos,usuarios_acesso_30d,pct_usuarios_acesso_30d,data_referencia
403,10,2.481389578163772,2026-10-02
403,13,3.225806451612903,2026-10-08
407,18,4.422604422604422,2026-10-22
